In [ ]:
import os, subprocess
from pathlib import Path
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["BLUFF_FAST_OBSERVATIONS"]="1"
os.environ["BLUFF_FAST_SNAPSHOTS"]="1"
work=Path("/kaggle/working/training");work.mkdir(exist_ok=True)
sources={'bluff_policy.py': '"""Compact learned Bluff action scorer using own cards and public evidence."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nRANKS = ("A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K")\nPLACE_FEATURES = ["batch_size", "truthful", "bluff", "bluff_pile", "bluff_call_rate",\n                  "bluff_finish", "remaining_groups", "remaining_largest_group", "claimed_rank_held"]\nCHALLENGE_FEATURES = ["bias", "revealed_bluff_rate", "rank_held", "claim_count", "pile_size",\n                      "opponent_finishing", "own_small_hand", "opponent_small_hand", "samples"]\nINITIAL_WEIGHTS = [4., 5., -2., -4., -8., -8., -1., .2, .5,\n                   -3.5, 7., .7, -.3, -.5, 1., -1., .5, 0.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(INITIAL_WEIGHTS):\n            raise ValueError("wrong feature count")\n        self.weights = [float(v) for v in weights]\n        self.processed = 0\n        self.claims = {}\n        self.stats = defaultdict(lambda: [0, 0])\n        self.our_outcomes = [0, 0]\n\n    def update(self, observation):\n        history = observation.get("public_history", [])\n        if len(history) < self.processed:\n            self.processed, self.claims, self.stats, self.our_outcomes = 0, {}, defaultdict(lambda: [0, 0]), [0, 0]\n        for event in history[self.processed:]:\n            kind = event.get("type")\n            identity = event.get("claim_id")\n            if kind == "claim":\n                self.claims[identity] = (event.get("seat"), event.get("count", 1))\n            elif kind in ("claim_accepted", "challenge_resolved"):\n                placer, count = self.claims.pop(identity, (event.get("placer", event.get("seat")), event.get("count", 1)))\n                if placer == observation["seat"]:\n                    self.our_outcomes[0] += 1\n                    self.our_outcomes[1] += int(kind == "challenge_resolved")\n                if kind == "challenge_resolved" and isinstance(event.get("truthful"), bool):\n                    stats = self.stats[(placer, count)]\n                    stats[0] += 1\n                    stats[1] += int(not event["truthful"])\n        self.processed = len(history)\n\n    def candidates(self, observation):\n        hand = observation["hand"]\n        held = Counter(c["rank"] for c in hand)\n        pile = min(observation.get("pile_size", 0), 40) / 20.\n        if observation["phase"] == "CHALLENGE":\n            claim = observation["pending_claim"]\n            count, rank, placer = claim["count"], claim["claimed_rank"], claim["seat"]\n            copies = 4 * observation.get("game_config", {}).get("decks", 1)\n            if count > copies - held[rank]:\n                return [({"type": "challenge"}, [0.] * len(self.weights))]\n            samples, bluffs = self.stats[(placer, count)]\n            posterior = (bluffs + .6) / (samples + 5.)\n            sizes = observation["hand_sizes"]\n            vector = [1., posterior, held[rank] / copies, count / 4., pile,\n                      float(claim.get("provisional_finish", False)), float(len(hand) <= 3),\n                      1. / max(1, sizes[placer]), min(samples, 20) / 20.]\n            return [({"type": "accept"}, [0.] * len(self.weights)),\n                    ({"type": "challenge"}, [0.] * len(PLACE_FEATURES) + vector)]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card["rank"]].append(card)\n        actions = []\n        # Safe truthful groups are always available. Constructed bluff candidates\n        # use only currently owned references; the count is the actual batch size.\n        for rank, cards in groups.items():\n            actions.append({"type": "place", "card_refs": [c["card_ref"] for c in cards[:4]], "claimed_rank": rank})\n        ordered = sorted(hand, key=lambda c: (held[c["rank"]], RANKS.index(c["rank"])))\n        for count in (2, 3):\n            if len(ordered) >= count:\n                for rank in groups:\n                    actions.append({"type": "place", "card_refs": [c["card_ref"] for c in ordered[:count]], "claimed_rank": rank})\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        by_ref = {c["card_ref"]: c for c in hand}\n        result = []\n        for action in actions:\n            refs = set(action["card_refs"])\n            rank = action["claimed_rank"]\n            truthful = all(by_ref[ref]["rank"] == rank for ref in refs)\n            remaining = Counter(c["rank"] for c in hand if c["card_ref"] not in refs)\n            vector = [len(refs), float(truthful), float(not truthful), float(not truthful) * pile,\n                      float(not truthful) * call_rate, float(not truthful and len(refs) == len(hand)),\n                      len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES)))\n        return result\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda candidate:\n                   sum(w * value for w, value in zip(self.weights, candidate[1])))[0]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-bluff-linear-v1", "features": PLACE_FEATURES + CHALLENGE_FEATURES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'bluff_policy_v2.py': '"""Learned Bluff scorer with conservative rank bounds and public caller evidence.\n\nInference reads only own hand/private history and public events. Unknown discarded\ncards decrement every rank bound, ensuring no hidden-hand identity is assumed.\n"""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom bluff_policy import Policy as V1, PLACE_FEATURES, CHALLENGE_FEATURES, INITIAL_WEIGHTS as V1_INITIAL, RANKS\n\nEXTRA_PLACE = [\'bluff_any_call_probability\', \'bluff_impossible_for_caller\',\n               \'truthful_turns_removed\', \'bluff_known_pile_fraction\',\n               \'truthful_finish\', \'bluff_no_impossibility_call_possible\']\nEXTRA_CHALLENGE = [\'placer_global_revealed_bluff_rate\', \'known_outside_rank_fraction\',\n                   \'known_pile_fraction\', \'claim_fraction_of_original_hand\',\n                   \'finishing_large_batch\', \'known_placer_rank_before_claim\']\nFEATURE_NAMES = PLACE_FEATURES + CHALLENGE_FEATURES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = [.5, 1., -.5, -2., -8., -3., -1., .2, .5] + V1_INITIAL[len(PLACE_FEATURES):] + [-6., -12., 3., -1., 4., 3., 1., 1., -.5, .5, .5, -.5]\n\n\ndef identity(card):\n    return card.get(\'deck\', 0), card[\'rank\'], card[\'suit\']\n\n\nclass Policy(V1):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid model weights\')\n        super().__init__(weights[:len(V1_INITIAL)])\n        self.weights = list(map(float, weights))\n        self._reset_evidence()\n\n    def _reset_evidence(self):\n        self.folded_history = []\n        self.known = defaultdict(Counter)\n        self.pile_known = {}\n        self.calls = defaultdict(lambda: [0, 0])\n        self.global_bluffs = defaultdict(lambda: [0, 0])\n        self.pending_meta = {}\n        self.active = []\n        self.before_claim = {}\n\n    def update(self, observation):\n        history = observation.get(\'public_history\', [])\n        if len(history) < len(self.folded_history) or history[:len(self.folded_history)] != self.folded_history:\n            self.processed = 0\n            self.claims = {}\n            self.stats = defaultdict(lambda: [0, 0])\n            self.our_outcomes = [0, 0]\n            self._reset_evidence()\n        super().update(observation)\n        own_places = {event[\'claim_id\']: event[\'cards\'] for event in observation.get(\'private_history\', []) if event.get(\'type\') == \'own_placement\'}\n        for event in history[len(self.folded_history):]:\n            kind, cid = event.get(\'type\'), event.get(\'claim_id\')\n            if kind == \'game_start\':\n                self.active = list(event[\'active_seats\'])\n            elif kind == \'claim\':\n                placer, count = event[\'seat\'], event[\'count\']\n                self.before_claim[cid] = dict(self.known[placer])\n                # The rank in an unverified claim is not evidence of actual cards.\n                for rank in list(self.known[placer]):\n                    self.known[placer][rank] = max(0, self.known[placer][rank] - count)\n                priority = sorted((s for s in self.active if s != placer), key=lambda s: (s - placer) % observation[\'game_config\'][\'players\'])\n                self.pending_meta[cid] = (placer, count, priority, bool(event.get(\'provisional_finish\')))\n                for card in own_places.get(cid, []):\n                    self.pile_known[identity(card)] = card\n            elif kind in (\'claim_accepted\', \'challenge_resolved\'):\n                placer, count, priority, finish = self.pending_meta.pop(cid)\n                selected = event.get(\'challenger\') if kind == \'challenge_resolved\' else None\n                # The first selected challenger is revealed. Earlier seats must\n                # have accepted; later choices remain unknown and are not counted.\n                for seat in priority:\n                    record = self.calls[(seat, count, finish)]\n                    record[0] += 1\n                    record[1] += int(seat == selected)\n                    if seat == selected:\n                        break\n                if kind == \'challenge_resolved\':\n                    record = self.global_bluffs[placer]\n                    record[0] += 1\n                    record[1] += int(not event[\'truthful\'])\n                    for card in event[\'revealed_batch\']:\n                        self.pile_known[identity(card)] = card\n            elif kind == \'pickup\':\n                recipient = event[\'recipient\']\n                self.known[recipient].update(card[\'rank\'] for card in self.pile_known.values())\n                self.pile_known = {}\n            elif kind == \'finish\':\n                self.active = [s for s in self.active if s != event[\'seat\']]\n        self.folded_history = list(history)\n\n    def caller_probability(self, observation, count, finish):\n        escape = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            # Beta prior avoids certainties from small or selectively observed samples.\n            escape *= 1. - (calls + 1.) / (samples + 5.)\n        return 1. - escape\n\n    def candidates(self, observation):\n        hand = observation[\'hand\']\n        held = Counter(c[\'rank\'] for c in hand)\n        copies = 4 * observation.get(\'game_config\', {}).get(\'decks\', 1)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            placer, rank, count = claim[\'seat\'], claim[\'claimed_rank\'], claim[\'count\']\n            outside = held[rank] + sum(counts[rank] for seat, counts in self.known.items() if seat not in (observation[\'seat\'], placer))\n            if count > copies - outside:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n            base = super().candidates(observation)\n            samples, bluffs = self.global_bluffs.get(placer, (0, 0))\n            extra = [(bluffs + .6) / (samples + 5.), outside / copies,\n                     len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     count / max(1, observation[\'hand_sizes\'][placer] + count),\n                     float(claim.get(\'provisional_finish\', False)) * count / 4.,\n                     self.before_claim.get(claim[\'claim_id\'], {}).get(rank, 0) / copies]\n            return [(a, v + [0.] * len(EXTRA_PLACE) + (extra if a[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE))) for a, v in base]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card[\'rank\']].append(card)\n        actions = [action for action, _ in super().candidates(observation)]\n        # Full truthful batches plus partial truthful batches and mixed shedding\n        # batches of up to four cards, including a possible finishing bluff.\n        for rank, cards in groups.items():\n            for count in range(1, min(4, len(cards)) + 1):\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:count]], \'claimed_rank\': rank})\n        ordered = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))\n        for count in range(2, min(4, len(hand)) + 1):\n            for rank in RANKS:\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in ordered[:count]], \'claimed_rank\': rank})\n        by_ref, seen, result = {c[\'card_ref\']: c for c in hand}, set(), []\n        pile = min(observation.get(\'pile_size\', 0), 40) / 20.\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        for action in actions:\n            refs, rank = set(action[\'card_refs\']), action[\'claimed_rank\']\n            key = tuple(sorted(refs)), rank\n            if key in seen:\n                continue\n            seen.add(key)\n            truthful = all(by_ref[r][\'rank\'] == rank for r in refs)\n            bluff = float(not truthful)\n            remaining = Counter(c[\'rank\'] for c in hand if c[\'card_ref\'] not in refs)\n            finish = not remaining\n            vector = [len(refs), float(truthful), bluff, bluff * pile, bluff * call_rate,\n                      float(bluff and finish), len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            impossible = any(self.known.get(s, {}).get(rank, 0) + len(refs) > copies for s in observation[\'active_seats\'] if s != observation[\'seat\'])\n            extra = [bluff * self.caller_probability(observation, len(refs), finish),\n                     bluff * impossible, sum((n + 3) // 4 for n in held.values()) - sum((n + 3) // 4 for n in remaining.values()),\n                     bluff * len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     float(truthful and finish), bluff * float(held[rank] >= len(refs))]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES) + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return result\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v2\', \'features\': FEATURE_NAMES,\n                                        \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v3.py': '"""Bluff v2 model with additional, sound public/own pile rank deductions.\n\nOlder unchallenged batches are never assumed truthful. These deductions use\nonly own placement cards and the last publicly revealed batch before pickup.\n"""\nfrom collections import Counter\nfrom bluff_policy_v2 import Policy as V2, FEATURE_NAMES, INITIAL_WEIGHTS\n\n\nclass Policy(V2):\n    def candidates(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            rank, count, placer = claim[\'claimed_rank\'], claim[\'count\'], claim[\'seat\']\n            # Known old pile cards, current own cards and lower bounds on other\n            # players\' hands are disjoint, so their sum is a conservative count.\n            pile_count = Counter(card[\'rank\'] for card in self.pile_known.values())[rank]\n            held_count = sum(card[\'rank\'] == rank for card in observation[\'hand\'])\n            other_count = sum(counts[rank] for seat, counts in self.known.items()\n                              if seat not in (observation[\'seat\'], placer))\n            copies = 4 * observation[\'game_config\'][\'decks\']\n            if count > copies - held_count - other_count - pile_count:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n        return super().candidates(observation)\n', 'bluff_policy_v4.py': '"""Linear Bluff scorer with learned interactions for public call-risk adaptation."""\nfrom collections import Counter\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v2 import FEATURE_NAMES as V2_NAMES, INITIAL_WEIGHTS as V2_INITIAL\n\nEXTRA_PLACE = [\'bluff_call_probability_batch\', \'bluff_call_probability_pile\',\n               \'bluff_call_probability_finish\', \'bluff_nearest_opponent_finish\',\n               \'truthful_call_probability_batch\', \'bluff_remaining_truthful_turns\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_finishing\', \'count_revealed_bluff_pile\',\n                   \'global_revealed_bluff_pile\', \'count_revealed_bluff_small_own_hand\',\n                   \'global_revealed_bluff_count\', \'opponent_hand_pace_relative_to_own\']\nFEATURE_NAMES = V2_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V2_INITIAL + [0.] * 12\n\n\nclass Policy(V3):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v4 model weights\')\n        super().__init__(weights[:len(V2_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        hand = observation[\'hand\']\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            global_rate = (bluffs + .6) / (samples + 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            count_rate = (bluffs + .6) / (samples + 5.)\n            extra = [global_rate * float(claim.get(\'provisional_finish\', False)),\n                     count_rate * pile, global_rate * pile,\n                     count_rate * float(len(hand) <= 3), global_rate * claim[\'count\'] / 4.,\n                     (len(hand) - observation[\'hand_sizes\'][claim[\'seat\']]) / max(1, len(hand))]\n            return [(action, vector[:len(V2_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     (extra if action[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE)))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in hand}\n        held = Counter(card[\'rank\'] for card in hand)\n        nearest = min(observation[\'hand_sizes\'][seat] for seat in observation[\'active_seats\']\n                      if seat != observation[\'seat\'])\n        results = []\n        risks = {}\n        for action, vector in candidates:\n            count = len(action[\'card_refs\']); finish = count == len(hand)\n            if count not in risks:\n                risks[count] = self.caller_probability(observation, count, finish)\n            risk = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            remaining = held.copy()\n            for ref in action[\'card_refs\']:\n                remaining[by_ref[ref][\'rank\']] -= 1\n            turns = sum((n + 3) // 4 for n in remaining.values() if n > 0)\n            extra = [bluff * risk * count / 4., bluff * risk * pile,\n                     bluff * risk * float(finish), bluff / max(1, nearest),\n                     (1. - bluff) * risk * count / 4., bluff * turns / 13.]\n            results.append((action, vector[:len(V2_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v4\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v5.py': '"""Public caller pooling and uncertainty features for a learned Bluff scorer.\n\nBeta summaries are modeling features, not calibrated certainty claims. Observed\nselected/earlier caller outcomes are pooled; later sealed votes stay unknown.\n"""\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v4 import Policy as V4, FEATURE_NAMES as V4_NAMES, INITIAL_WEIGHTS as V4_INITIAL\n\nEXTRA_PLACE = [\'bluff_pooled_caller_probability\', \'bluff_pooled_caller_uncertainty\',\n               \'bluff_specific_caller_uncertainty\', \'bluff_sparse_own_outcomes\',\n               \'truthful_pooled_caller_probability_batch\', \'bluff_pooled_caller_probability_pile\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_uncertainty\', \'count_revealed_bluff_uncertainty\']\nFEATURE_NAMES = V4_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V4_INITIAL + [0.] * 8\n\n\ndef beta_summary(samples, successes, prior_success=1., prior_total=5.):\n    a = successes + prior_success\n    b = samples - successes + prior_total - prior_success\n    return a / (a + b), math.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.)))\n\n\nclass Policy(V4):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v5 model weights\')\n        super().__init__(weights[:len(V4_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def caller_summaries(self, observation, count=None, finish=None):\n        escape_mean = escape_upper = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            if count is None:\n                samples = calls = 0\n                for (caller, _, _), record in self.calls.items():\n                    if caller == seat:\n                        samples += record[0]; calls += record[1]\n            else:\n                samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            mean, deviation = beta_summary(samples, calls)\n            escape_mean *= 1. - mean\n            escape_upper *= 1. - min(1., mean + deviation)\n        return 1. - escape_mean, escape_mean - escape_upper\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            _, global_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            _, count_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            return [(action, vector[:len(V4_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     ([global_uncertainty, count_uncertainty] if action[\'type\'] == \'challenge\' else [0., 0.]))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in observation[\'hand\']}\n        pooled, pooled_uncertainty = self.caller_summaries(observation)\n        sparse = float(self.our_outcomes[0] <= 2)\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        risks = {}\n        results = []\n        for action, vector in candidates:\n            count = len(action[\'card_refs\'])\n            if count not in risks:\n                risks[count] = self.caller_summaries(observation, count, count == len(observation[\'hand\']))\n            _, specific_uncertainty = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            extra = [bluff * pooled, bluff * pooled_uncertainty, bluff * specific_uncertainty,\n                     bluff * sparse, (1. - bluff) * pooled * count / 4., bluff * pooled * pile]\n            results.append((action, vector[:len(V4_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v5\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_mlp_policy.py': '"""Tiny residual neural Bluff scorer with standard-library inference."""\nimport math\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\n\n\nclass Policy(Features):\n    def __init__(self, document):\n        if document.get(\'format\') != \'triad-bluff-residual-mlp-v1\' or document.get(\'features\') != FEATURE_NAMES:\n            raise ValueError(\'incompatible neural Bluff schema\')\n        base = document[\'base_weights\']\n        super().__init__(base)\n        self.hidden = document[\'hidden_weight\']; self.bias = document[\'hidden_bias\']\n        self.output = document[\'output_weight\']; self.output_bias = float(document[\'output_bias\'])\n        self.means = document[\'feature_mean\']; self.scales = document[\'feature_scale\']\n        width = len(self.hidden)\n        if not 1 <= width <= 128 or len(self.bias) != width or len(self.output) != width:\n            raise ValueError(\'invalid hidden layer shape\')\n        if any(len(row) != len(FEATURE_NAMES) for row in self.hidden) or len(self.means) != len(FEATURE_NAMES) or len(self.scales) != len(FEATURE_NAMES):\n            raise ValueError(\'invalid neural input shape\')\n        parameters = base + self.bias + self.output + self.means + self.scales + [self.output_bias]\n        parameters += [weight for row in self.hidden for weight in row]\n        if not all(math.isfinite(float(weight)) for weight in parameters) or any(scale <= 0. for scale in self.scales):\n            raise ValueError(\'nonfinite neural parameters or invalid feature scale\')\n\n    def score_features(self, features):\n        normalized = [(value - mean) / scale for value, mean, scale in zip(features, self.means, self.scales)]\n        residual = self.output_bias\n        for row, bias, output in zip(self.hidden, self.bias, self.output):\n            hidden = max(0., bias + sum(weight * value for weight, value in zip(row, normalized)))\n            residual += output * hidden\n        return sum(weight * value for weight, value in zip(self.weights, features)) + residual\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda item: self.score_features(item[1]))[0]\n\n\nclass FastPolicy(Policy):\n    """Training-only NumPy batching; near ties use exact deployment scoring."""\n    def __init__(self, document):\n        super().__init__(document)\n        import numpy as np\n        self.np = np\n        self.matrix = np.asarray(self.hidden, dtype=np.float64)\n        self.hidden_bias = np.asarray(self.bias, dtype=np.float64)\n        self.output_array = np.asarray(self.output, dtype=np.float64)\n        self.base_array = np.asarray(self.weights, dtype=np.float64)\n        self.mean_array = np.asarray(self.means, dtype=np.float64)\n        self.scale_array = np.asarray(self.scales, dtype=np.float64)\n        # These exported parameters are immutable configuration, never actor\n        # memory. Freezing them allows exact snapshot sharing during rollouts.\n        self.weights = tuple(self.weights)\n        self.hidden = tuple(tuple(row) for row in self.hidden)\n        self.bias = tuple(self.bias); self.output = tuple(self.output)\n        self.means = tuple(self.means); self.scales = tuple(self.scales)\n        for matrix in (self.matrix, self.hidden_bias, self.output_array,\n                       self.base_array, self.mean_array, self.scale_array):\n            matrix.flags.writeable = False\n\n\n    def __deepcopy__(self, memo):\n        # Copy public/private actor memory, and share only frozen configuration.\n        # This avoids repeatedly copying thousands of constant network weights.\n        import copy\n        clone = type(self).__new__(type(self))\n        memo[id(self)] = clone\n        configuration = {\'np\', \'weights\', \'hidden\', \'bias\', \'output\', \'output_bias\',\n                         \'means\', \'scales\', \'matrix\', \'hidden_bias\', \'output_array\',\n                         \'base_array\', \'mean_array\', \'scale_array\'}\n        for name, value in self.__dict__.items():\n            setattr(clone, name, value if name in configuration else copy.deepcopy(value, memo))\n        return clone\n\n    def act(self, observation):\n        self.update(observation)\n        candidates = self.candidates(observation)\n        matrix = self.np.asarray([vector for _, vector in candidates], dtype=self.np.float64)\n        scores = matrix @ self.base_array + self.output_bias\n        scores += self.np.maximum(0., ((matrix - self.mean_array) / self.scale_array) @ self.matrix.T + self.hidden_bias) @ self.output_array\n        largest = float(scores.max())\n        band = 1e-8 * max(1., abs(largest))\n        finalists = [index for index, value in enumerate(scores) if value >= largest - band]\n        chosen = max(finalists, key=lambda index: self.score_features(candidates[index][1]))\n        return candidates[chosen][0]\n', 'bluff_training_observation.py': '"""Training-only observation shortcut for verified read-only simulator actors.\n\nAll fields still come from the SDK\'s filtered observe(). The engine is not\nadvanced while histories are temporarily detached. Only the requested seat\'s\nprivate history is returned. Never import this helper into an inference image.\n"""\nfrom triad_bluff import engine\n\n\ndef observe_read_only(state, seat):\n    public = state.public_history\n    private = state.private_history[seat]\n    try:\n        state.public_history = []\n        state.private_history[seat] = []\n        observation = engine.observe(state, seat)\n    finally:\n        state.public_history = public\n        state.private_history[seat] = private\n    observation[\'public_history\'] = public\n    observation[\'private_history\'] = private\n    return observation\n', 'bluff_training_snapshot.py': '"""Training-only simulator clone sharing verified immutable prior records.\n\nThe pinned SDK freezes physical Card/GameConfig values and only appends to\npublic/private histories and debug traces. Verified read-only actors never\nchange those prior records. Mutable containers, current claims, RNG state and\nactor evidence still deepcopy normally. Full simulator state remains label-only.\n"""\nimport copy\n\n\ndef clone_snapshot(snapshot):\n    state, _ = snapshot\n    if not type(state.config).__dataclass_params__.frozen:\n        raise TypeError(\'snapshot sharing requires the pinned frozen GameConfig\')\n    memo = {id(state.config): state.config}\n    for cards in (*state.hands, state.pile, state.pending_batch):\n        for card in cards:\n            if not type(card).__dataclass_params__.frozen:\n                raise TypeError(\'snapshot sharing requires immutable physical cards\')\n            memo[id(card)] = card\n    for records in (state.public_history, *state.private_history, state.trace):\n        for record in records:\n            memo[id(record)] = record\n    return copy.deepcopy(snapshot, memo)\n', 'train_bluff_league.py': '"""Train against diverse Bluff policies with paired seat blocks and held-out tests."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get(\'BLUFF_SDK\', ROOT.parent / \'bluffSDK\'))\nsys.path.insert(0, str(SDK / \'local_evaluator/src\'))\nfrom triad_bluff import engine\nfrom triad_bluff.baselines import HeuristicAgent, RandomAgent, TruthfulAgent, AggressiveChallengeAgent\nfrom bluff_policy import Policy as V1, INITIAL_WEIGHTS as V1_INITIAL, RANKS\nfrom bluff_policy_v2 import Policy as V2, INITIAL_WEIGHTS, FEATURE_NAMES\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v4 import Policy as V4, INITIAL_WEIGHTS as V4_INITIAL\nfrom bluff_policy_v5 import Policy as V5, INITIAL_WEIGHTS as V5_INITIAL\nfrom bluff_training_observation import observe_read_only\n\n\nclass Starter(TruthfulAgent):\n    def act(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            return super().act(observation)\n        from collections import defaultdict\n        groups = defaultdict(list)\n        for card in observation[\'hand\']:\n            groups[card[\'rank\']].append(card)\n        rank, cards = max(groups.items(), key=lambda item: (len(item[1]), item[0]))\n        return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:4]], \'claimed_rank\': rank}\n\n\nclass SDKCompetitive:\n    """The SDK\'s supplied adaptive strategy, used only as a development rival."""\n    def __init__(self):\n        import importlib.util\n        path = SDK / \'agents/bluff-agent/src/strategy.py\'\n        spec = importlib.util.spec_from_file_location(\'sdk_competitive_strategy\', path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        self.select_action = module.select_action\n        self.state = None\n\n    def act(self, observation):\n        action, self.state = self.select_action(observation, self.state)\n        return action\n\n\nclass StochasticBluffer(TruthfulAgent):\n    def __init__(self, seed, bluff_rate, call_rate):\n        self.rng = random.Random(seed)\n        self.bluff_rate, self.call_rate = bluff_rate, call_rate\n\n    def act(self, observation):\n        truthful = super().act(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if truthful[\'type\'] == \'challenge\':\n                return truthful\n            finishing = observation[\'pending_claim\'].get(\'provisional_finish\', False)\n            probability = min(.95, self.call_rate + .2 * finishing)\n            return {\'type\': \'challenge\' if self.rng.random() < probability else \'accept\'}\n        hand = observation[\'hand\']\n        if len(hand) > 1 and self.rng.random() < self.bluff_rate:\n            from collections import Counter\n            held = Counter(c[\'rank\'] for c in hand)\n            count = min(4, len(hand)) if len(hand) <= 4 else self.rng.choice((2, 3, 4))\n            chosen = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))[:count]\n            # Owning copies of the claimed rank makes impossible calls less likely.\n            claimed = max(RANKS, key=lambda rank: held[rank])\n            return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in chosen], \'claimed_rank\': claimed}\n        return truthful\n\n\ndef make_policy(spec, seed):\n    kind = spec[\'kind\']\n    if kind == \'mlp\':\n        from bluff_mlp_policy import FastPolicy\n        return FastPolicy(spec[\'model\'])\n    if kind == \'v1\':\n        return V1(spec[\'weights\'])\n    if kind == \'v2\':\n        return V2(spec[\'weights\'])\n    if kind == \'v3\':\n        return V3(spec[\'weights\'])\n    if kind == \'v4\':\n        return V4(spec[\'weights\'])\n    if kind == \'v5\':\n        return V5(spec[\'weights\'])\n    if kind == \'bluffer\':\n        return StochasticBluffer(seed, spec[\'bluff_rate\'], spec[\'call_rate\'])\n    if kind == \'starter\':\n        return Starter()\n    if kind == \'competitive\':\n        return SDKCompetitive()\n    if kind == \'truthful\':\n        return TruthfulAgent()\n    if kind == \'random\':\n        return RandomAgent(seed)\n    if kind == \'aggressive\':\n        return AggressiveChallengeAgent(seed)\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals, horizon=1000):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, rival in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(rival, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=horizon), seed)\n    while state.outcome is None and seat not in state.finished_set:\n        decision = engine.next_decision(state)\n        observation = observe_read_only(state, decision.seat) if os.environ.get(\'BLUFF_FAST_OBSERVATIONS\') == \'1\' else engine.observe(state, decision.seat)\n        payload = actors[decision.seat].act(observation)\n        action = engine.decode_action(state, decision, payload)\n        if not engine.step(state, decision, action).ok:\n            raise RuntimeError(\'policy produced an illegal action\')\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    return state.outcome.points[seat], 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode, *options = job\n    horizon = options[0] if options else 1000\n    points, wins = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x424C554646)\n        if mode == \'league\':\n            rivals = [rng.choice(pool) for _ in range(3)]\n        elif mode == \'bluffer\':\n            rivals = [{\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}] * 3\n        else:\n            rivals = [{\'kind\': mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals, horizon) for seat in range(4)]\n        points.append(sum(p for p, _ in outcomes) / 4.)\n        wins.append(sum(w for _, w in outcomes) / 4.)\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(seeds), \'block_points\': points, \'block_wins\': wins}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith(\'block_\')}\n\n\ndef parallel_evaluate(pool, candidate, seeds, reference, mode):\n    """Distribute validation seed blocks instead of leaving other workers idle."""\n    chunks = [seeds[i:i + 16] for i in range(0, len(seeds), 16)]\n    parts = list(pool.map(evaluate_job, [(candidate, chunk, reference, mode) for chunk in chunks]))\n    points = [point for part in parts for point in part[\'block_points\']]\n    wins = [win for part in parts for win in part[\'block_wins\']]\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(points), \'block_points\': points, \'block_wins\': wins}\n\n\ndef paired_interval(candidate, reference, seed=20261014):\n    differences = np.asarray(candidate[\'block_points\']) - np.asarray(reference[\'block_points\'])\n    rng = np.random.default_rng(seed)\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {\'mean_gain\': float(differences.mean()), \'ci95\': [float(v) for v in np.quantile(means, [.025, .975])],\n            \'unit\': \'points per game\', \'sampling_unit\': \'four-seat seed block\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path)\n    parser.add_argument(\'--reference-policy\', type=Path)\n    parser.add_argument(\'--policy-kind\', choices=(\'v2\', \'v3\', \'v4\', \'v5\'), default=\'v2\')\n    parser.add_argument(\'--fast-observations\', action=\'store_true\')\n    parser.add_argument(\'--stress-opponents\', action=\'store_true\')\n    parser.add_argument(\'--selection-seed-start\', type=int, default=8_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=9_000_000)\n    parser.add_argument(\'--train-seed-start\', type=int, default=50_000)\n    parser.add_argument(\'--iterations\', type=int, default=30)\n    parser.add_argument(\'--population\', type=int, default=32)\n    parser.add_argument(\'--train-blocks\', type=int, default=8)\n    parser.add_argument(\'--train-horizon\', type=int, default=200)\n    parser.add_argument(\'--validation-blocks\', type=int, default=64)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--workers\', type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument(\'--seed\', type=int, default=20261016)\n    parser.add_argument(\'--output\', type=Path, default=ROOT / \'bluff-league-output\')\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers, args.train_horizon) < 1 or args.train_horizon > 1000:\n        parser.error(\'counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    if args.fast_observations:\n        os.environ[\'BLUFF_FAST_OBSERVATIONS\'] = \'1\'\n    incumbent_document = json.loads(args.incumbent.read_text()) if args.incumbent else {\'weights\': V1_INITIAL}\n    weights = incumbent_document[\'weights\']\n    incumbent_kind = incumbent_document.get(\'training\', {}).get(\'policy_kind\', \'v2\' if len(weights) == len(FEATURE_NAMES) else \'v4\' if len(weights) == len(V4_INITIAL) else \'v5\' if len(weights) == len(V5_INITIAL) else \'v1\')\n    incumbent = {\'kind\': incumbent_kind, \'weights\': weights}\n    initial_weights = V5_INITIAL if args.policy_kind == \'v5\' else V4_INITIAL if args.policy_kind == \'v4\' else INITIAL_WEIGHTS\n    mean = np.asarray(weights + initial_weights[len(weights):], dtype=float)\n    best, sigma = mean.copy(), np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    cautious = list(INITIAL_WEIGHTS); cautious[2] -= 8.; cautious[4] -= 8.\n    daring = list(INITIAL_WEIGHTS); daring[2] += 4.; daring[5] += 4.\n    reference = [{\'kind\': k} for k in (\'starter\', \'heuristic\', \'truthful\', \'aggressive\', \'random\', \'competitive\')] + [\n        incumbent, {\'kind\': \'v2\', \'weights\': INITIAL_WEIGHTS}, {\'kind\': \'v2\', \'weights\': cautious},\n        {\'kind\': \'v2\', \'weights\': daring}] + [\n        {\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.reference_policy:\n        reference_document = json.loads(args.reference_policy.read_text())\n        reference.append({\'kind\': reference_document.get(\'training\', {}).get(\'policy_kind\', \'v1\' if len(reference_document[\'weights\']) == len(V1_INITIAL) else \'v2\'),\n                          \'weights\': reference_document[\'weights\']})\n    if args.stress_opponents:\n        reference.extend({\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c}\n                         for b, c in ((.1, .35), (.65, .35), (.95, .0), (.95, .35)))\n    archive = list(reference)\n    selection_seeds = list(range(args.selection_seed_start, args.selection_seed_start + args.validation_blocks))\n    history, started = [], time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        baseline = parallel_evaluate(pool, incumbent, selection_seeds, reference, \'league\')\n        initial = parallel_evaluate(pool, {\'kind\': args.policy_kind, \'weights\': best.tolist()}, selection_seeds, reference, \'league\')\n        selected_score = initial[\'points\']\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(args.train_seed_start + generation * args.train_blocks, args.train_seed_start + (generation + 1) * args.train_blocks))\n            # Fast training probes use a shorter cap; model selection and final\n            # reporting always use the real 1,000-turn cap. Short-cap results are\n            # approximate training rewards, never evidence of event performance.\n            scores = list(pool.map(evaluate_job, [({\'kind\':args.policy_kind,\'weights\':w.tolist()},seeds,archive,\'league\',args.train_horizon) for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i][\'points\'], scores[i][\'win_rate\']), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {\'kind\':args.policy_kind,\'weights\':candidate.tolist()}\n            validation = parallel_evaluate(pool, spec, selection_seeds, reference, \'league\')\n            if validation[\'points\'] > selected_score:\n                best, selected_score = candidate.copy(), validation[\'points\']\n            if generation % 4 == 3:\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {\'generation\':generation+1,\'train\':summarize(scores[ranked[0]]),\'validation\':summarize(validation),\n                      \'paired_validation_gain\':paired_interval(validation,baseline),\'selected_points\':selected_score,\n                      \'archive_size\':len(archive),\'elapsed_seconds\':round(time.time()-started,1)}\n            history.append(record); print(json.dumps(record),flush=True)\n            policy_class = V5 if args.policy_kind == \'v5\' else V4 if args.policy_kind == \'v4\' else V2\n            policy_class(best).save(args.output/\'bluff-policy-v2.json\',{\'method\':\'archive-league cross-entropy policy optimization\',\n                \'fast_observations\': args.fast_observations, \'stress_opponents\': args.stress_opponents,\n                \'policy_kind\': args.policy_kind, \'selection_seed_start\': args.selection_seed_start,\n                \'test_seed_start\': args.test_seed_start, \'train_seed_start\': args.train_seed_start,\n                \'seed\':args.seed,\'generations\':generation+1,\'population\':args.population,\n                \'train_blocks\':args.train_blocks,\'train_horizon\':args.train_horizon,\'evaluation_horizon\':1000,\'validation_blocks\':args.validation_blocks,\n                \'selected_validation_points\':selected_score,\'incumbent_validation_points\':baseline[\'points\'],\n                \'sdk_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\'})\n            (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n            (args.output/\'archive.json\').write_text(json.dumps(archive,indent=2)+\'\\n\')\n        seeds = list(range(args.test_seed_start,args.test_seed_start+args.test_blocks))\n        chosen = {\'kind\':args.policy_kind,\'weights\':best.tolist()}\n        modes = (\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\',\'competitive\')\n        results = list(pool.map(evaluate_job,[(spec,seeds,reference,mode) for mode in modes for spec in (chosen,incumbent)]))\n        report = {mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                        \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'collect_bluff_counterfactual.py': '"""Paired offline policy-improvement returns; deployment features are own/public.\n\nThe dealt simulator hands are held fixed within each labeled state. Opponent\nrandom choices are averaged using common trial seeds across candidate actions.\nWhole-game, phase-stratified reservoir sampling covers later/endgame decisions.\nThere is no draw deck to shuffle in Bluff. Hidden simulator state is confined to\ntraining returns and never appears in the exported feature vectors.\n"""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport time\nfrom collections import Counter\nimport numpy as np\nfrom train_bluff_league import engine, make_policy\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\nfrom bluff_training_observation import observe_read_only\n\n\ndef reward(state, seat):\n    if seat in state.finished_order:\n        return float(state.config.position_points[state.finished_order.index(seat)])\n    return float(state.outcome.points[seat])\n\n\ndef advance(state, actors):\n    decision = engine.next_decision(state)\n    observation = observe_read_only(state, decision.seat)\n    action = actors[decision.seat].act(observation)\n    if not engine.step(state, decision, engine.decode_action(state, decision, action)).ok:\n        raise RuntimeError(\'rollout actor produced an illegal action\')\n\n\ndef copy_snapshot(snapshot):\n    if os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\':\n        from bluff_training_snapshot import clone_snapshot\n        return clone_snapshot(snapshot)\n    return copy.deepcopy(snapshot)\n\n\ndef rollout(snapshot, seat, selected, trial_seed):\n    state, actors = copy_snapshot(snapshot)\n    # Preserve each actor\'s public/private memory at the sampled decision while\n    # integrating future stochastic choices. Each action gets the same streams.\n    for player, actor in enumerate(actors):\n        if hasattr(actor, \'rng\'):\n            actor.rng = random.Random(trial_seed * 10 + player)\n    decision = engine.next_decision(state)\n    if decision.seat != seat:\n        raise RuntimeError(\'sampled decision is for a different player\')\n    if not engine.step(state, decision, engine.decode_action(state, decision, selected)).ok:\n        raise RuntimeError(\'counterfactual action is not legal at sampled state\')\n    while state.outcome is None and seat not in state.finished_set:\n        advance(state, actors)\n    return reward(state, seat)\n\n\ndef checkpoint_spec(document):\n    if document.get(\'format\') == \'triad-bluff-residual-mlp-v1\':\n        return {\'kind\': \'mlp\', \'model\': document}\n    kind = document.get(\'training\', {}).get(\'policy_kind\',\n        \'v2\' if len(document[\'weights\']) == 30 else \'v1\')\n    return {\'kind\': kind, \'weights\': document[\'weights\']}\n\n\ndef pool_specs(incumbent, previous):\n    return [checkpoint_spec(incumbent), checkpoint_spec(previous),\n            *({\'kind\': kind} for kind in (\'starter\', \'competitive\', \'random\', \'aggressive\', \'heuristic\')),\n            *({\'kind\': \'bluffer\', \'bluff_rate\': bluff, \'call_rate\': call}\n              for bluff, call in ((.1, .0), (.1, .35), (.35, .1), (.65, .35), (.95, .0), (.95, .35)))]\n\n\nSOURCE_REGIMES = (\'league\', \'neural_selfplay\', \'v2_selfplay\', \'competitive\',\n                  \'truthful\', \'bluffer\', \'frequent_call\')\n\n\ndef source_specs(regime, rng, incumbent, previous):\n    if regime == \'league\':\n        return [rng.choice(pool_specs(incumbent, previous)) for _ in range(4)]\n    if regime == \'neural_selfplay\':\n        rival = checkpoint_spec(incumbent)\n    elif regime == \'v2_selfplay\':\n        rival = checkpoint_spec(previous)\n    elif regime in (\'competitive\', \'truthful\'):\n        rival = {\'kind\': regime}\n    elif regime == \'bluffer\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}\n    elif regime == \'frequent_call\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .65, \'call_rate\': .35}\n    else:\n        raise ValueError(\'unknown source composition\')\n    return [rival] * 4\n\n\ndef collect_job(job):\n    seed, incumbent, previous, samples_per_phase, trials, max_actions, regime = job\n    rng = random.Random(seed ^ 0x424C554646)\n    own = seed % 4\n    specs = source_specs(regime, rng, incumbent, previous)\n    specs[own] = checkpoint_spec(incumbent)\n    actors = [make_policy(spec, seed * 10 + player) for player, spec in enumerate(specs)]\n    state = engine.new_game(engine.GameConfig(max_turns=1000), seed)\n    reservoirs = {\'PLACE\': [], \'CHALLENGE\': []}\n    eligible = Counter()\n    initial = incumbent.get(\'base_weights\', incumbent.get(\'weights\'))\n    feature_weights = initial + [0.] * (len(FEATURE_NAMES) - len(initial))\n    while state.outcome is None and own not in state.finished_set:\n        decision = engine.next_decision(state)\n        if decision.seat == own:\n            observation = observe_read_only(state, own)\n            policy = Features(feature_weights); policy.update(observation)\n            if len(policy.candidates(observation)) > 1:\n                phase = observation[\'phase\']\n                eligible[phase] += 1\n                index = rng.randrange(eligible[phase])\n                if index < samples_per_phase:\n                    snapshot = copy_snapshot((state, actors))\n                    if len(reservoirs[phase]) < samples_per_phase:\n                        reservoirs[phase].append(snapshot)\n                    else:\n                        reservoirs[phase][index] = snapshot\n        advance(state, actors)\n    rows = []\n    for phase in (\'PLACE\', \'CHALLENGE\'):\n        for snapshot in reservoirs[phase]:\n            original, _ = snapshot\n            observation = observe_read_only(original, own)\n            policy = Features(feature_weights); policy.update(observation)\n            candidates = policy.candidates(observation)\n            baseline = make_policy(specs[own], seed * 10 + own).act(observation)\n            preferred = next(item for item in candidates if item[0] == baseline) if any(item[0] == baseline for item in candidates) else candidates[0]\n            # Identical model inputs cannot be distinguished by the scorer;\n            # avoid spending rollout compute on exact feature aliases.\n            representatives = {tuple(preferred[1]): preferred}\n            for candidate in candidates:\n                representatives.setdefault(tuple(candidate[1]), candidate)\n            selected = list(representatives.values())\n            if len(selected) > max_actions:\n                selected = [preferred] + rng.sample([item for item in selected if item is not preferred], max_actions - 1)\n            trial_values = [[rollout(snapshot, own, action, seed * 100_000 + len(rows) * 1000 + trial)\n                             for trial in range(trials)] for action, _ in selected]\n            features = [vector for _, vector in selected]\n            targets = [sum(values) / trials for values in trial_values]\n            rows.append((features, targets, trial_values, seed, phase, len(observation[\'hand\']), regime))\n    return rows\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path, required=True)\n    parser.add_argument(\'--previous\', type=Path, required=True)\n    parser.add_argument(\'--games\', type=int, default=2000)\n    parser.add_argument(\'--samples-per-phase\', type=int, default=2)\n    parser.add_argument(\'--trials\', type=int, default=2)\n    parser.add_argument(\'--max-actions\', type=int, default=10)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--seed-start\', type=int, default=50_000_000)\n    parser.add_argument(\'--source-compositions\', choices=(\'league\', \'balanced\'), default=\'league\')\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    if min(args.games, args.samples_per_phase, args.trials, args.workers) < 1 or args.max_actions < 2:\n        parser.error(\'positive counts and at least two candidate actions required\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    incumbent = json.loads(args.incumbent.read_text()); previous = json.loads(args.previous.read_text())\n    features, targets, trial_targets, offsets, game_ids = [], [], [], [0], []\n    phases = Counter(); hands = Counter(); compositions = Counter(); source_regimes = []; started = time.time()\n    jobs = [(seed, incumbent, previous, args.samples_per_phase, args.trials, args.max_actions,\n             SOURCE_REGIMES[(seed - args.seed_start) % len(SOURCE_REGIMES)] if args.source_compositions == \'balanced\' else \'league\')\n            for seed in range(args.seed_start, args.seed_start + args.games)]\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for index, rows in enumerate(pool.map(collect_job, jobs)):\n            for vectors, values, trials, seed, phase, hand, regime in rows:\n                features.extend(vectors); targets.extend(values); trial_targets.extend(trials)\n                offsets.append(len(features)); game_ids.append(seed); phases[phase] += 1; hands[hand] += 1\n                source_regimes.append(SOURCE_REGIMES.index(regime)); compositions[regime] += 1\n            if (index + 1) % 100 == 0:\n                print(json.dumps({\'games\': index + 1, \'states\': len(game_ids), \'actions\': len(features),\n                                  \'seconds\': round(time.time() - started, 1)}), flush=True)\n    np.savez_compressed(args.output / \'counterfactual.npz\', features=np.asarray(features, dtype=np.float32),\n                        targets=np.asarray(targets, dtype=np.float32), trial_targets=np.asarray(trial_targets, dtype=np.float32),\n                        offsets=np.asarray(offsets, dtype=np.int64), game_ids=np.asarray(game_ids, dtype=np.int64),\n                        source_regimes=np.asarray(source_regimes, dtype=np.int8))\n    metadata = {\'method\': \'paired one-step policy improvement, phase-stratified whole-game reservoirs\',\n                \'input_features\': FEATURE_NAMES, \'feature_policy\': \'v5\', \'games\': args.games,\n                \'states\': len(game_ids), \'actions\': len(features), \'trials\': args.trials,\n                \'samples_per_phase\': args.samples_per_phase, \'max_actions\': args.max_actions,\n                \'seed_start\': args.seed_start, \'incumbent\': incumbent, \'opponent_pool\': pool_specs(incumbent, previous),\n                \'sdk_game_commit\': \'96da8042c285af4e4f43655f2ad99592bd49b8e6\',\n                \'incumbent_policy_kind\': checkpoint_spec(incumbent)[\'kind\'],\n                \'source_compositions\': args.source_compositions, \'source_regime_names\': SOURCE_REGIMES,\n                \'source_game_composition_histogram\': dict(Counter(job[-1] for job in jobs)),\n                \'sampled_state_composition_histogram\': dict(compositions),\n                \'fast_snapshots\': os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\',\n                \'hidden_state_use\': \'training returns only; dealt hands fixed within state, common future actor seeds across actions\',\n                \'future_draw_pile_shuffle\': False, \'own_hand_size_histogram\': dict(sorted(hands.items())),\n                \'phase_histogram\': dict(phases), \'seconds\': round(time.time() - started, 1)}\n    (args.output / \'metadata.json\').write_text(json.dumps(metadata, indent=2) + \'\\n\')\n    print(json.dumps({key: value for key, value in metadata.items() if key not in (\'incumbent\', \'input_features\', \'opponent_pool\')}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name,text in sources.items(): (work/name).write_text(text)
(work/'incumbent.json').write_text('{\n  "format": "triad-bluff-residual-mlp-v1",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim",\n    "bluff_call_probability_batch",\n    "bluff_call_probability_pile",\n    "bluff_call_probability_finish",\n    "bluff_nearest_opponent_finish",\n    "truthful_call_probability_batch",\n    "bluff_remaining_truthful_turns",\n    "global_revealed_bluff_finishing",\n    "count_revealed_bluff_pile",\n    "global_revealed_bluff_pile",\n    "count_revealed_bluff_small_own_hand",\n    "global_revealed_bluff_count",\n    "opponent_hand_pace_relative_to_own",\n    "bluff_pooled_caller_probability",\n    "bluff_pooled_caller_uncertainty",\n    "bluff_specific_caller_uncertainty",\n    "bluff_sparse_own_outcomes",\n    "truthful_pooled_caller_probability_batch",\n    "bluff_pooled_caller_probability_pile",\n    "global_revealed_bluff_uncertainty",\n    "count_revealed_bluff_uncertainty"\n  ],\n  "base_weights": [\n    0.11889829486608505,\n    1.4551690816879272,\n    -0.8614801168441772,\n    0.12563253939151764,\n    -2.55124568939209,\n    -2.6591808795928955,\n    0.4780709743499756,\n    1.6731796264648438,\n    0.8349596261978149,\n    -0.9919783473014832,\n    1.7952346801757812,\n    -2.062161922454834,\n    -1.0631221532821655,\n    -1.4293476343154907,\n    -2.5744378566741943,\n    -1.0431323051452637,\n    0.9109983444213867,\n    1.1962584257125854,\n    -2.227351665496826,\n    -2.295670509338379,\n    0.8897631168365479,\n    -0.3832564353942871,\n    1.4892596006393433,\n    2.4285433292388916,\n    0.8895594477653503,\n    -0.11950855702161789,\n    -0.31598660349845886,\n    0.15420950949192047,\n    0.7154812216758728,\n    -1.7789300680160522,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n  ],\n  "feature_mean": [\n    2.130405902862549,\n    0.2556110918521881,\n    0.5673422813415527,\n    0.1414773166179657,\n    0.20922617614269257,\n    0.044980913400650024,\n    0.22203241288661957,\n    0.4362170994281769,\n    0.304308146238327,\n    0.08852332085371017,\n    0.012027798220515251,\n    0.005555760581046343,\n    0.05282813683152199,\n    0.03254607692360878,\n    0.002083986531943083,\n    0.015454696491360664,\n    0.015533286146819592,\n    0.002624368527904153,\n    0.28184229135513306,\n    0.06834737956523895,\n    1.5457278490066528,\n    0.023674404248595238,\n    0.010973202995955944,\n    0.09815023094415665,\n    0.015177818946540356,\n    0.007621304597705603,\n    0.008625890128314495,\n    0.020141232758760452,\n    0.0009267285349778831,\n    0.006892831530421972,\n    0.21310904622077942,\n    0.05682198330760002,\n    0.020169535651803017,\n    0.12666580080986023,\n    0.05104479193687439,\n    0.13413675129413605,\n    0.00024044152814894915,\n    0.0043762726709246635,\n    0.005239620339125395,\n    0.0022578288335353136,\n    0.00981908105313778,\n    -0.10914473235607147,\n    0.2997722029685974,\n    0.0871744155883789,\n    0.12125638127326965,\n    0.23309296369552612,\n    0.05302620306611061,\n    0.044763777405023575,\n    0.011351659893989563,\n    0.011535020545125008\n  ],\n  "feature_scale": [\n    1.3542357683181763,\n    0.43625375628471375,\n    0.4955291450023651,\n    0.32548826932907104,\n    0.23007379472255707,\n    0.20728932321071625,\n    0.20233742892742157,\n    0.3010976016521454,\n    0.2976628541946411,\n    0.28402191400527954,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.18657560646533966,\n    0.15651710331439972,\n    0.10000000149011612,\n    0.12336929887533188,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.2647395730018616,\n    0.2523006200790405,\n    1.334463357925415,\n    0.10000000149011612,\n    0.1041841208934784,\n    0.29747799038887024,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.21329738199710846,\n    0.12353038787841797,\n    0.10000000149011612,\n    0.18938058614730835,\n    0.10285001248121262,\n    0.1730906069278717,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.9756709337234497,\n    0.3055936396121979,\n    0.10000000149011612,\n    0.10987535119056702,\n    0.4228348731994629,\n    0.11198851466178894,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612\n  ],\n  "hidden_weight": [\n    [\n      -0.19053113460540771,\n      0.15571263432502747,\n      -0.1011505052447319,\n      0.19476446509361267,\n      0.27500608563423157,\n      0.15307991206645966,\n      0.3815036416053772,\n      0.4381793439388275,\n      -0.22956830263137817,\n      0.027035346254706383,\n      -0.021941421553492546,\n      0.004426548257470131,\n      0.04070810601115227,\n      -0.05979225039482117,\n      0.0010170531459152699,\n      -0.08468963205814362,\n      0.03122447244822979,\n      0.04913467541337013,\n      0.017885252833366394,\n      0.07352987676858902,\n      0.07204055041074753,\n      -0.6565521359443665,\n      0.23869308829307556,\n      -0.137938991189003,\n      0.22352753579616547,\n      0.07389921694993973,\n      -0.021464945748448372,\n      -0.07632283121347427,\n      -0.0017953533679246902,\n      -0.15402238070964813,\n      -0.2351987063884735,\n      0.2819085419178009,\n      -0.04682714119553566,\n      0.3633670508861542,\n      0.1471453309059143,\n      -0.030505336821079254,\n      0.11275077611207962,\n      -0.04455846548080444,\n      -0.19067446887493134,\n      -0.02123730443418026,\n      0.15277132391929626,\n      -0.0441616028547287,\n      0.19499239325523376,\n      0.005214580800384283,\n      -0.16005682945251465,\n      0.41382917761802673,\n      -0.3361380696296692,\n      0.03683564066886902,\n      0.17921903729438782,\n      0.09538088738918304\n    ],\n    [\n      -0.34688928723335266,\n      -0.0341823436319828,\n      0.08536931127309799,\n      -0.15115806460380554,\n      0.2503475546836853,\n      -0.015907486900687218,\n      0.6435881853103638,\n      0.10033245384693146,\n      0.2746313512325287,\n      -0.13110876083374023,\n      -0.2760956585407257,\n      0.004355510231107473,\n      0.11898820102214813,\n      -0.0032388579566031694,\n      0.1219732016324997,\n      -0.09869767725467682,\n      -0.027204176411032677,\n      -0.41678473353385925,\n      0.26750725507736206,\n      -0.011986590921878815,\n      0.05444285273551941,\n      0.28975367546081543,\n      0.05204622447490692,\n      -0.5399481058120728,\n      -0.10123363882303238,\n      0.03145816922187805,\n      0.08416414260864258,\n      0.2687167227268219,\n      0.013803120702505112,\n      0.27541759610176086,\n      -0.3447941243648529,\n      0.18976999819278717,\n      0.16411860287189484,\n      -0.0076088146306574345,\n      -0.14338168501853943,\n      0.16006532311439514,\n      -0.009708402678370476,\n      0.07059653103351593,\n      -0.1445803940296173,\n      -0.09821895509958267,\n      -0.19294242560863495,\n      0.016644466668367386,\n      0.3165804147720337,\n      -0.35061442852020264,\n      0.046103350818157196,\n      -0.020881682634353638,\n      -0.44616249203681946,\n      -0.184276282787323,\n      -0.04051925987005234,\n      -0.076309435069561\n    ],\n    [\n      -0.43836283683776855,\n      -0.03827773407101631,\n      0.02417719177901745,\n      0.0012864257441833615,\n      -0.18788781762123108,\n      0.05437812581658363,\n      -0.048713840544223785,\n      0.1781936138868332,\n      0.36789923906326294,\n      0.15997500717639923,\n      0.08273530751466751,\n      0.11687079071998596,\n      -0.03200904652476311,\n      -0.10258307307958603,\n      -0.044549256563186646,\n      -0.07501626014709473,\n      0.10152506083250046,\n      0.42922788858413696,\n      0.24868355691432953,\n      0.25955644249916077,\n      0.2103431671857834,\n      -0.06754159182310104,\n      0.10614077001810074,\n      -0.3674386739730835,\n      0.03684944659471512,\n      -0.011080002412199974,\n      -0.17129723727703094,\n      -0.07273056358098984,\n      0.012710507027804852,\n      0.0012870589271187782,\n      0.040439359843730927,\n      0.007053954526782036,\n      -0.1431591808795929,\n      0.7334985733032227,\n      -0.020398594439029694,\n      0.07002000510692596,\n      -0.1414639800786972,\n      -0.014410188421607018,\n      0.0591629222035408,\n      -0.02386775054037571,\n      0.1051902174949646,\n      0.06887765228748322,\n      -0.14827775955200195,\n      0.2279590368270874,\n      -0.06324269622564316,\n      0.19206000864505768,\n      -0.15030698478221893,\n      0.04369090497493744,\n      -0.09575971961021423,\n      0.10184834897518158\n    ],\n    [\n      -0.8009151220321655,\n      0.042741164565086365,\n      0.2561957538127899,\n      -0.03245321288704872,\n      0.2126467525959015,\n      -0.3626656234264374,\n      -0.08744355291128159,\n      0.252923846244812,\n      -0.2406282126903534,\n      -0.2533353567123413,\n      -0.3336438238620758,\n      -0.49595001339912415,\n      0.03433312103152275,\n      0.1379004865884781,\n      -0.5763046741485596,\n      -0.4408527612686157,\n      -0.17515550553798676,\n      0.08869050443172455,\n      0.26932889223098755,\n      -0.33251070976257324,\n      -0.2426517903804779,\n      0.0012413441436365247,\n      0.11090754717588425,\n      0.05123414471745491,\n      -0.24921815097332,\n      -0.4750996530056,\n      0.48253777623176575,\n      -0.027838952839374542,\n      -0.38995078206062317,\n      -0.31328147649765015,\n      -0.3602568507194519,\n      -0.19446098804473877,\n      -0.543244481086731,\n      -0.11715645343065262,\n      0.13799040019512177,\n      -0.1073513776063919,\n      -0.37774690985679626,\n      0.022098375484347343,\n      -0.13915301859378815,\n      -0.4281318783760071,\n      -0.2979890704154968,\n      0.10360624641180038,\n      0.3499918580055237,\n      0.13586987555027008,\n      0.13379111886024475,\n      -0.06283661723136902,\n      0.1657276302576065,\n      0.3291209638118744,\n      -0.15946771204471588,\n      -0.14842531085014343\n    ],\n    [\n      -0.4666611850261688,\n      -0.21241676807403564,\n      0.1910749077796936,\n      0.06911717355251312,\n      0.23509050905704498,\n      0.31656408309936523,\n      0.2708587646484375,\n      0.49206769466400146,\n      0.24235671758651733,\n      0.026647981256246567,\n      0.1187470406293869,\n      -0.14169251918792725,\n      -0.08341462910175323,\n      0.12279897183179855,\n      0.05708643049001694,\n      -0.23100155591964722,\n      0.0009216552716679871,\n      -0.23335391283035278,\n      -0.03357965499162674,\n      0.3808215260505676,\n      0.23694391548633575,\n      -0.15176722407341003,\n      0.14328934252262115,\n      -0.06307930499315262,\n      0.1969938725233078,\n      0.33162984251976013,\n      -0.10945877432823181,\n      0.34658071398735046,\n      0.11664565652608871,\n      0.10710567981004715,\n      -0.3654142916202545,\n      0.28199678659439087,\n      0.089230976998806,\n      0.3477048873901367,\n      0.03435347229242325,\n      0.17952503263950348,\n      -0.005079449154436588,\n      0.3451724946498871,\n      0.2539703845977783,\n      0.31034937500953674,\n      0.21049605309963226,\n      0.5691021680831909,\n      -0.04177477955818176,\n      0.14757639169692993,\n      0.46376973390579224,\n      0.05747527629137039,\n      0.18095451593399048,\n      -0.25173884630203247,\n      0.18668290972709656,\n      0.14818932116031647\n    ],\n    [\n      -0.7069453001022339,\n      0.10696735233068466,\n      -0.1955098658800125,\n      0.04555068910121918,\n      0.3874591588973999,\n      0.20683079957962036,\n      -0.446833997964859,\n      0.29962068796157837,\n      -0.034959275275468826,\n      0.20622512698173523,\n      0.12322287261486053,\n      0.025812890380620956,\n      0.03312835097312927,\n      -0.08387616276741028,\n      0.015942800790071487,\n      0.16793064773082733,\n      0.23285695910453796,\n      0.048181504011154175,\n      0.04697374999523163,\n      0.07852131873369217,\n      -0.1740342527627945,\n      -0.21954193711280823,\n      0.04958309605717659,\n      -0.4439425766468048,\n      0.10293136537075043,\n      -0.044991955161094666,\n      0.09410923719406128,\n      -0.0233745314180851,\n      -0.08290598541498184,\n      0.06586933881044388,\n      -0.13631898164749146,\n      0.054769959300756454,\n      0.2936391532421112,\n      -0.4733392000198364,\n      0.025694748386740685,\n      -0.30140426754951477,\n      0.10703320801258087,\n      -0.00779837416484952,\n      0.07616543024778366,\n      -0.020140083506703377,\n      0.06963186711072922,\n      -0.03516196087002754,\n      0.28908097743988037,\n      -0.1399356573820114,\n      0.448266863822937,\n      0.4235385060310364,\n      -0.3639507591724396,\n      -0.09979131817817688,\n      0.018416909500956535,\n      0.01727212220430374\n    ],\n    [\n      -0.03547343611717224,\n      0.042599186301231384,\n      -0.13445666432380676,\n      0.27867192029953003,\n      -0.1933528035879135,\n      0.14748847484588623,\n      0.23424869775772095,\n      0.6576681137084961,\n      -0.2924177944660187,\n      0.0798158124089241,\n      0.10613710433244705,\n      0.015439657494425774,\n      0.14775849878787994,\n      0.09118485450744629,\n      -0.12247393280267715,\n      0.22114013135433197,\n      0.07602106779813766,\n      0.29813095927238464,\n      -0.31005844473838806,\n      0.1560666412115097,\n      -0.2705908715724945,\n      -0.09668433666229248,\n      0.14817866683006287,\n      0.12874020636081696,\n      0.23493953049182892,\n      0.06945419311523438,\n      -0.07678230106830597,\n      0.11114998906850815,\n      0.059269193559885025,\n      0.13170039653778076,\n      -0.32500356435775757,\n      0.21914531290531158,\n      0.2580072581768036,\n      -0.18756340444087982,\n      0.60267573595047,\n      0.09367770701646805,\n      0.002875487320125103,\n      0.031517673283815384,\n      -0.012696266174316406,\n      0.12384146451950073,\n      0.10011958330869675,\n      -0.13896508514881134,\n      -0.06378152966499329,\n      0.12267380952835083,\n      0.07059703767299652,\n      -0.03849215805530548,\n      0.3489634692668915,\n      0.17285385727882385,\n      0.34713345766067505,\n      0.10579388588666916\n    ],\n    [\n      -0.3740462064743042,\n      -0.14343155920505524,\n      0.13711591064929962,\n      0.21937614679336548,\n      0.805578887462616,\n      -0.48606735467910767,\n      -0.6440969109535217,\n      0.008069561794400215,\n      0.03596563637256622,\n      0.06841330230236053,\n      0.09124680608510971,\n      0.20227183401584625,\n      -0.1476958841085434,\n      -0.10490380972623825,\n      -0.19745688140392303,\n      -0.3149966597557068,\n      0.0642014816403389,\n      0.09790126234292984,\n      -0.3586054742336273,\n      -0.14411096274852753,\n      -0.3387010395526886,\n      -0.461421936750412,\n      0.012739426456391811,\n      0.07183930277824402,\n      0.07319998741149902,\n      0.06467419862747192,\n      -0.02730027213692665,\n      0.0177640188485384,\n      0.009762308560311794,\n      0.18767277896404266,\n      -0.25299814343452454,\n      0.004556243307888508,\n      -0.4331885874271393,\n      0.6903507709503174,\n      0.09927846491336823,\n      0.11865057051181793,\n      -0.19630247354507446,\n      0.03672018647193909,\n      0.006273719482123852,\n      -0.27636194229125977,\n      0.10056620836257935,\n      -0.2750764787197113,\n      -0.2743605375289917,\n      0.11487728357315063,\n      -0.21548666059970856,\n      -0.0849001407623291,\n      0.1967046856880188,\n      0.3868160545825958,\n      -0.11838560551404953,\n      0.10246492177248001\n    ],\n    [\n      -0.15618306398391724,\n      -0.0239772517234087,\n      0.04755749553442001,\n      0.010224930010735989,\n      0.11524950712919235,\n      -0.014770879410207272,\n      -0.06294459104537964,\n      -0.031985167413949966,\n      0.032739587128162384,\n      0.09298791736364365,\n      -0.11481371521949768,\n      -0.15526078641414642,\n      -0.022801987826824188,\n      -0.15966346859931946,\n      0.05545642971992493,\n      0.21264657378196716,\n      -0.08094839006662369,\n      0.4150295555591583,\n      0.038763172924518585,\n      0.14816109836101532,\n      -0.10293320566415787,\n      0.05836347118020058,\n      -0.0792263001203537,\n      -0.32350730895996094,\n      0.1665249615907669,\n      -0.04150903970003128,\n      -0.05239039659500122,\n      -0.08471309393644333,\n      0.121266208589077,\n      -0.0014270353130996227,\n      -0.05903973802924156,\n      -0.02083747088909149,\n      0.06164424121379852,\n      0.12489137798547745,\n      -0.10239112377166748,\n      0.11750168353319168,\n      -0.14621612429618835,\n      0.04299991577863693,\n      0.04783204570412636,\n      0.15346753597259521,\n      0.014543632045388222,\n      0.10683470964431763,\n      -0.03323371335864067,\n      -0.003854575101286173,\n      -0.19214366376399994,\n      -0.03526030108332634,\n      -0.04858594015240669,\n      0.10026992112398148,\n      0.007083422504365444,\n      -0.1050834059715271\n    ],\n    [\n      -0.2340119630098343,\n      -0.1311962902545929,\n      0.012483648024499416,\n      0.08511599898338318,\n      -0.044495388865470886,\n      0.20780839025974274,\n      0.20153303444385529,\n      0.2624434232711792,\n      0.32059332728385925,\n      0.08335617929697037,\n      0.04421668499708176,\n      -0.015539539977908134,\n      0.007245282176882029,\n      0.18704229593276978,\n      -0.26119303703308105,\n      -0.43681997060775757,\n      0.03169839456677437,\n      -0.3552597463130951,\n      0.1179923489689827,\n      -0.10485481470823288,\n      0.23826444149017334,\n      -0.25279805064201355,\n      0.058726225048303604,\n      -0.31730422377586365,\n      0.014462308958172798,\n      -0.00023079095990397036,\n      0.1388983428478241,\n      0.12865278124809265,\n      -0.12762616574764252,\n      0.005441761109977961,\n      -0.1515406370162964,\n      -0.051951657980680466,\n      0.17133884131908417,\n      0.685310423374176,\n      -0.00957696232944727,\n      0.038983482867479324,\n      -0.1642482876777649,\n      0.1801123023033142,\n      0.03890753909945488,\n      -0.14680218696594238,\n      0.06145242601633072,\n      1.042846918106079,\n      -0.042383112013339996,\n      -0.15215317904949188,\n      -0.24692203104496002,\n      0.18917889893054962,\n      -0.14350120723247528,\n      0.26828792691230774,\n      0.23018144071102142,\n      0.10932588577270508\n    ],\n    [\n      0.3322123885154724,\n      -0.18821783363819122,\n      0.0765753984451294,\n      0.23367373645305634,\n      0.3780375123023987,\n      0.1865251213312149,\n      0.3168398439884186,\n      -0.8197890520095825,\n      0.11141742765903473,\n      -0.03586093708872795,\n      -0.0836518257856369,\n      0.19181115925312042,\n      0.22029082477092743,\n      -0.024909770116209984,\n      -0.21364426612854004,\n      0.04330916702747345,\n      -0.02714921161532402,\n      -0.6437059640884399,\n      0.10146825015544891,\n      0.2402789294719696,\n      0.27162525057792664,\n      -0.2091243714094162,\n      0.011973648332059383,\n      -0.018197134137153625,\n      -0.09150644391775131,\n      0.1643635779619217,\n      -0.02910521812736988,\n      0.11285676062107086,\n      -0.033838752657175064,\n      -0.22987478971481323,\n      0.13096605241298676,\n      0.12056934833526611,\n      0.024511314928531647,\n      -0.2536150813102722,\n      -0.19454994797706604,\n      0.34864115715026855,\n      -0.10269317030906677,\n      0.15696042776107788,\n      -0.32188767194747925,\n      -0.09561910480260849,\n      0.060371577739715576,\n      0.08191504329442978,\n      -0.8268782496452332,\n      -0.2580358386039734,\n      0.3688259422779083,\n      -0.48169219493865967,\n      -0.049497392028570175,\n      0.29896190762519836,\n      -0.026430204510688782,\n      0.008099472150206566\n    ],\n    [\n      -0.029509885236620903,\n      -0.34242621064186096,\n      0.16561579704284668,\n      -0.010497413575649261,\n      0.04179583117365837,\n      0.10720138251781464,\n      0.04319336637854576,\n      0.02871839515864849,\n      0.023956257849931717,\n      -0.01933148317039013,\n      -0.16761919856071472,\n      -0.09857246279716492,\n      -0.22773438692092896,\n      0.13983239233493805,\n      -0.2308560311794281,\n      -0.2406291514635086,\n      -0.06087779253721237,\n      0.06693939119577408,\n      0.17468194663524628,\n      0.1563374251127243,\n      -0.1310565173625946,\n      0.49474406242370605,\n      -0.15331414341926575,\n      -0.32666221261024475,\n      -0.20615436136722565,\n      -0.21729524433612823,\n      0.20773786306381226,\n      -0.3822798728942871,\n      -0.04398162290453911,\n      -0.1281927078962326,\n      0.007586438674479723,\n      -0.611744225025177,\n      -0.05213972553610802,\n      -0.09962411969900131,\n      -0.6330392956733704,\n      0.23169295489788055,\n      -0.2639612555503845,\n      0.13328424096107483,\n      -0.06009092554450035,\n      -0.1239374503493309,\n      -0.3297577500343323,\n      0.10535461455583572,\n      0.23223823308944702,\n      -0.09761741012334824,\n      -0.3502937853336334,\n      -0.18727360665798187,\n      -0.29612648487091064,\n      -0.056469522416591644,\n      -0.10573189705610275,\n      -0.14346741139888763\n    ],\n    [\n      -0.26724305748939514,\n      0.08889330178499222,\n      0.008113402873277664,\n      -0.27317148447036743,\n      0.2011149823665619,\n      -0.04121546074748039,\n      -0.22013381123542786,\n      -0.797607421875,\n      0.36875659227371216,\n      -0.058561183512210846,\n      0.08153904974460602,\n      -0.007255932316184044,\n      -0.0984342098236084,\n      0.057313233613967896,\n      -0.0735824853181839,\n      -0.26956233382225037,\n      0.017771027982234955,\n      -0.08869937062263489,\n      -0.005857986398041248,\n      -0.25606125593185425,\n      -0.09762841463088989,\n      -0.33106791973114014,\n      -0.06652376800775528,\n      -0.12843550741672516,\n      -0.08270307630300522,\n      -0.046221524477005005,\n      -0.09148037433624268,\n      0.07982458919286728,\n      0.03203823044896126,\n      0.016645997762680054,\n      0.039965804666280746,\n      0.2062731832265854,\n      0.11458174884319305,\n      0.18539603054523468,\n      -0.022299231961369514,\n      -0.49218472838401794,\n      -0.09138567000627518,\n      -0.004584163893014193,\n      0.05923668295145035,\n      -0.29008257389068604,\n      0.10724376887083054,\n      -0.07637747377157211,\n      -0.23378099501132965,\n      0.05910341441631317,\n      0.203413188457489,\n      -0.09736248105764389,\n      0.2546629011631012,\n      0.09849297255277634,\n      -0.24182625114917755,\n      -0.07155266404151917\n    ],\n    [\n      -0.3199852705001831,\n      -0.04902562126517296,\n      -0.11015060544013977,\n      0.13555565476417542,\n      -0.1786220222711563,\n      0.09948953241109848,\n      0.14066071808338165,\n      -0.21072794497013092,\n      -0.5030348896980286,\n      -0.08439727872610092,\n      -0.2530125081539154,\n      -0.3614713251590729,\n      -0.05258870869874954,\n      -0.10137499868869781,\n      -0.22220778465270996,\n      0.32201874256134033,\n      0.0799582377076149,\n      0.15970627963542938,\n      0.0795905590057373,\n      0.27575740218162537,\n      -0.044520117342472076,\n      -0.18729595839977264,\n      -0.11038057506084442,\n      -0.3810097873210907,\n      -0.4145146608352661,\n      -0.056077029556035995,\n      -0.14889715611934662,\n      -0.11386126279830933,\n      -0.4162250757217407,\n      -0.043115030974149704,\n      -0.08273240923881531,\n      -0.021875593811273575,\n      0.010167443193495274,\n      0.3572275638580322,\n      0.09942388534545898,\n      0.3671188950538635,\n      -0.3396758437156677,\n      -0.25534388422966003,\n      -0.4077712297439575,\n      0.06318295001983643,\n      -0.32920631766319275,\n      -0.07883904874324799,\n      -0.010043187066912651,\n      0.07424140721559525,\n      0.05924276262521744,\n      -0.2450912594795227,\n      0.04750494286417961,\n      0.019476082175970078,\n      -0.2556335926055908,\n      -0.15605509281158447\n    ],\n    [\n      0.6340359449386597,\n      0.04145973548293114,\n      -0.07292081415653229,\n      0.07251635938882828,\n      -0.08202191442251205,\n      0.04293441027402878,\n      0.5576285719871521,\n      0.23475027084350586,\n      0.44962626695632935,\n      0.006248448044061661,\n      0.06210228055715561,\n      -0.2050963193178177,\n      0.03397884964942932,\n      -0.022366419434547424,\n      0.08456797897815704,\n      0.0763671025633812,\n      0.10751130431890488,\n      -0.08908049017190933,\n      -0.44752663373947144,\n      -0.2895398437976837,\n      0.33557626605033875,\n      -0.272480845451355,\n      0.09224913269281387,\n      0.35606077313423157,\n      0.2651411294937134,\n      0.4373738169670105,\n      0.3549002408981323,\n      0.08648732304573059,\n      0.14063279330730438,\n      -0.06875208020210266,\n      0.1601409614086151,\n      0.041062165051698685,\n      0.04832049459218979,\n      0.1375073343515396,\n      -0.16349495947360992,\n      -0.20357580482959747,\n      0.04089963436126709,\n      -0.06695489585399628,\n      0.12484778463840485,\n      0.03442877158522606,\n      0.15138018131256104,\n      -0.3351605534553528,\n      -0.2105487734079361,\n      0.008606968447566032,\n      -0.06378739327192307,\n      -0.318213552236557,\n      0.036649588495492935,\n      0.0718257948756218,\n      0.05229468643665314,\n      0.10798918455839157\n    ],\n    [\n      -0.4880310297012329,\n      -0.290319561958313,\n      0.18820735812187195,\n      -0.06892693042755127,\n      0.20368099212646484,\n      -0.26760879158973694,\n      0.31209075450897217,\n      -0.593944787979126,\n      0.498654305934906,\n      0.10167057812213898,\n      0.14594915509223938,\n      0.10385118424892426,\n      -0.30846941471099854,\n      -0.16828879714012146,\n      -0.1857483834028244,\n      -0.4659593105316162,\n      -0.09148602187633514,\n      0.9138979911804199,\n      0.05556275323033333,\n      -0.04497753083705902,\n      -0.07722266763448715,\n      -0.08538047969341278,\n      -0.1086912751197815,\n      -0.18396438658237457,\n      -0.0110703120008111,\n      -0.2346513569355011,\n      0.6007078886032104,\n      -0.4091869592666626,\n      -0.10134308785200119,\n      0.08310243487358093,\n      -0.33675816655158997,\n      0.280028373003006,\n      -0.03388960659503937,\n      -0.2295314520597458,\n      -0.2214173972606659,\n      0.1346883475780487,\n      -0.15148119628429413,\n      -0.11658339947462082,\n      -0.3361017405986786,\n      -0.3612913191318512,\n      -0.09712904691696167,\n      0.024362584576010704,\n      -0.09650429338216782,\n      -0.24035996198654175,\n      0.10614266246557236,\n      0.09804969280958176,\n      -0.5469701886177063,\n      -0.13870614767074585,\n      -0.027438627555966377,\n      0.026141872629523277\n    ],\n    [\n      0.07229895144701004,\n      0.03486057370901108,\n      -0.19355346262454987,\n      0.29655587673187256,\n      0.13598886132240295,\n      0.10626787692308426,\n      0.09095804393291473,\n      0.31509214639663696,\n      -0.13798213005065918,\n      0.23979517817497253,\n      0.276653528213501,\n      -0.03847532346844673,\n      0.32585328817367554,\n      0.05657418817281723,\n      0.09893937408924103,\n      0.145035058259964,\n      0.029836229979991913,\n      0.2778482437133789,\n      -0.35207322239875793,\n      0.08971316367387772,\n      0.4157799780368805,\n      0.028854133561253548,\n      -0.10814930498600006,\n      0.055448174476623535,\n      0.22193656861782074,\n      -0.0042461915872991085,\n      0.030554668977856636,\n      0.16198888421058655,\n      -0.035144224762916565,\n      0.05954234302043915,\n      -0.6285920143127441,\n      0.18410919606685638,\n      0.1915920525789261,\n      -0.5593177080154419,\n      -0.1998709887266159,\n      -0.3633144795894623,\n      -0.005100379232317209,\n      0.22080577909946442,\n      0.07437482476234436,\n      0.16885313391685486,\n      0.27040478587150574,\n      -0.29884520173072815,\n      0.18099215626716614,\n      0.2769649624824524,\n      0.051309648901224136,\n      0.05014529824256897,\n      0.4624188542366028,\n      0.24487924575805664,\n      0.14120706915855408,\n      0.2307230830192566\n    ],\n    [\n      -0.28400999307632446,\n      -0.31002405285835266,\n      -0.012761787511408329,\n      0.14118428528308868,\n      0.4426450729370117,\n      0.1171816810965538,\n      0.10751694440841675,\n      -0.6219719052314758,\n      0.035196222364902496,\n      -0.1460052877664566,\n      -0.022354193031787872,\n      -0.04214084893465042,\n      -0.04056141525506973,\n      -0.0316203348338604,\n      0.058338381350040436,\n      -0.07202573865652084,\n      -0.16142955422401428,\n      -0.12001245468854904,\n      0.3406381607055664,\n      0.0032692679669708014,\n      -0.36121946573257446,\n      -0.22993430495262146,\n      -0.06961605697870255,\n      -0.808266282081604,\n      -0.10942716896533966,\n      -0.10159999877214432,\n      -0.10182453691959381,\n      -0.23560519516468048,\n      -0.14857430756092072,\n      0.0991126224398613,\n      0.08501067012548447,\n      0.12716858088970184,\n      -0.10174532979726791,\n      0.01189819723367691,\n      -0.0013789650984108448,\n      0.2006131261587143,\n      0.03138815984129906,\n      -0.18153460323810577,\n      -0.03544573113322258,\n      -0.20842579007148743,\n      -0.1990332305431366,\n      0.017326844856142998,\n      0.08995715528726578,\n      -0.29164257645606995,\n      -0.054569680243730545,\n      0.07508371770381927,\n      0.1298971027135849,\n      0.11354754865169525,\n      -0.11951977014541626,\n      -0.007311002351343632\n    ],\n    [\n      -0.313959002494812,\n      0.02367100678384304,\n      -0.23750454187393188,\n      0.19636744260787964,\n      -0.4092922508716583,\n      0.3455292284488678,\n      0.1998608410358429,\n      0.22046808898448944,\n      -0.35031044483184814,\n      0.03594677150249481,\n      0.12130099534988403,\n      0.04169800877571106,\n      0.2498067170381546,\n      -0.0372464656829834,\n      0.1514752060174942,\n      -0.011637207120656967,\n      0.17202650010585785,\n      0.10219906270503998,\n      -0.44123736023902893,\n      0.20308686792850494,\n      -0.13556955754756927,\n      0.0338115394115448,\n      0.07485858350992203,\n      -0.441572904586792,\n      0.19151411950588226,\n      0.06156781688332558,\n      -0.09438305348157883,\n      -0.032099466770887375,\n      0.12792031466960907,\n      -0.09593029320240021,\n      -0.14163430035114288,\n      0.21665973961353302,\n      0.2212689220905304,\n      -0.09448543190956116,\n      -0.09020856767892838,\n      -0.09307733178138733,\n      -0.11562037467956543,\n      0.21669070422649384,\n      0.2042226493358612,\n      0.10017234086990356,\n      0.0006136708543635905,\n      0.2547128200531006,\n      -0.24404467642307281,\n      -0.17294526100158691,\n      -0.012948770076036453,\n      0.18893244862556458,\n      -0.06888674944639206,\n      0.09324324131011963,\n      0.1519479602575302,\n      0.11566980183124542\n    ],\n    [\n      -0.4943947196006775,\n      -0.37880223989486694,\n      0.12686514854431152,\n      -0.028622223064303398,\n      0.17918363213539124,\n      -0.13050785660743713,\n      0.2026994228363037,\n      -0.17935694754123688,\n      0.4380630552768707,\n      -0.11766725033521652,\n      -0.129194974899292,\n      -0.04541458934545517,\n      0.1604277342557907,\n      -0.1607377976179123,\n      -0.04632454365491867,\n      -0.011155758053064346,\n      -0.13659162819385529,\n      -1.0188244581222534,\n      0.37685921788215637,\n      -0.18514999747276306,\n      0.07884564250707626,\n      0.1309960037469864,\n      -0.02880169078707695,\n      -0.28296878933906555,\n      -0.004682627506554127,\n      -0.030437076464295387,\n      -0.015656590461730957,\n      0.21033556759357452,\n      -0.1351899802684784,\n      -0.2936747372150421,\n      -0.44261977076530457,\n      0.15120938420295715,\n      -0.21807026863098145,\n      -0.34597519040107727,\n      -0.0027242463547736406,\n      0.11175087839365005,\n      -0.13817018270492554,\n      -0.11875613778829575,\n      -0.05769561976194382,\n      0.12040862441062927,\n      0.00770455552265048,\n      0.07549260556697845,\n      -0.2092660665512085,\n      -0.1579517424106598,\n      0.35385575890541077,\n      -0.2981831431388855,\n      -0.14543303847312927,\n      -0.41573572158813477,\n      -0.1754993051290512,\n      -0.25732412934303284\n    ],\n    [\n      -0.5843906998634338,\n      0.023006102070212364,\n      -0.2868063449859619,\n      0.19544805586338043,\n      -0.2703016400337219,\n      -0.4782143831253052,\n      0.15871690213680267,\n      -0.5789566040039062,\n      0.27165892720222473,\n      0.16037949919700623,\n      0.15952976047992706,\n      -0.4202682077884674,\n      -0.061652787029743195,\n      0.007442873436957598,\n      -0.14205460250377655,\n      0.12480668723583221,\n      0.10434304177761078,\n      0.33649593591690063,\n      0.006205507088452578,\n      0.37024521827697754,\n      -0.1162598580121994,\n      0.2969197630882263,\n      -0.1422189474105835,\n      -0.04484979435801506,\n      0.33930695056915283,\n      -0.3059912323951721,\n      -0.26080235838890076,\n      -0.08195023238658905,\n      -0.2389908730983734,\n      0.2225254625082016,\n      -0.020228303968906403,\n      0.08015476167201996,\n      -0.20816384255886078,\n      0.5272433757781982,\n      -0.24563837051391602,\n      -0.231959730386734,\n      -0.4278993308544159,\n      -0.19001170992851257,\n      -0.06070360913872719,\n      0.01738264597952366,\n      -0.032029006630182266,\n      -0.32012397050857544,\n      0.7366903424263,\n      -0.35473912954330444,\n      0.04068528115749359,\n      0.10522628575563431,\n      0.4234182834625244,\n      0.4338505268096924,\n      0.2428039014339447,\n      0.15285803377628326\n    ],\n    [\n      0.32526499032974243,\n      0.0700555145740509,\n      0.16584300994873047,\n      0.14409296214580536,\n      -0.12014655023813248,\n      -0.1813012957572937,\n      0.23767665028572083,\n      -0.5436683297157288,\n      0.04718820005655289,\n      -0.05421098321676254,\n      0.0914592370390892,\n      -0.03039310686290264,\n      -0.21307942271232605,\n      -0.21871115267276764,\n      0.24239540100097656,\n      -0.1697438806295395,\n      0.15428747236728668,\n      -1.0806773900985718,\n      -0.14046365022659302,\n      -0.26631084084510803,\n      0.0949084609746933,\n      0.576619029045105,\n      0.02062864601612091,\n      -0.3112156391143799,\n      0.029294926673173904,\n      0.30035170912742615,\n      0.19895866513252258,\n      -0.058695610612630844,\n      -0.013687225058674812,\n      0.09398889541625977,\n      -0.00045071024214848876,\n      0.04743901267647743,\n      -0.22651807963848114,\n      -0.48898157477378845,\n      -0.22358489036560059,\n      0.39652758836746216,\n      0.3221066892147064,\n      -0.03893650323152542,\n      0.18594510853290558,\n      -0.13303305208683014,\n      -0.03647095710039139,\n      0.07256828993558884,\n      -0.13758498430252075,\n      0.11072687059640884,\n      0.29907891154289246,\n      -0.04181970655918121,\n      -0.07504820823669434,\n      0.08294575661420822,\n      -0.058195117861032486,\n      0.07449038326740265\n    ],\n    [\n      -0.5233381986618042,\n      0.0008672281401231885,\n      -0.03805981203913689,\n      0.21171528100967407,\n      0.23875588178634644,\n      0.04219716042280197,\n      0.5243920683860779,\n      0.24311195313930511,\n      0.1120820939540863,\n      0.14307865500450134,\n      0.09491016715765,\n      0.18101878464221954,\n      0.08173248171806335,\n      0.11200929433107376,\n      -0.04997217655181885,\n      0.13585078716278076,\n      0.03232493996620178,\n      0.08581267297267914,\n      -0.07864118367433548,\n      0.1779915690422058,\n      -0.058080434799194336,\n      0.03136714547872543,\n      0.11727762222290039,\n      -0.5342357158660889,\n      0.10215844959020615,\n      0.031690146774053574,\n      -0.12930142879486084,\n      0.03609846532344818,\n      -0.02158038690686226,\n      0.051236532628536224,\n      -0.36258170008659363,\n      0.31467580795288086,\n      0.03029535710811615,\n      -0.24494363367557526,\n      0.02350180596113205,\n      -0.026698840782046318,\n      0.05107489600777626,\n      0.12503203749656677,\n      0.02067023143172264,\n      0.1583503931760788,\n      0.07459188997745514,\n      -0.08375407755374908,\n      -0.13252781331539154,\n      0.08350452780723572,\n      0.028414392843842506,\n      0.24695615470409393,\n      0.07464083284139633,\n      -0.0393829308450222,\n      0.16417503356933594,\n      0.03753133490681648\n    ],\n    [\n      -0.25898006558418274,\n      0.025537917390465736,\n      0.05677321180701256,\n      -0.03887172043323517,\n      0.3491106331348419,\n      0.005391890183091164,\n      0.14687418937683105,\n      -0.7723749876022339,\n      0.34060534834861755,\n      -0.04562126472592354,\n      -0.00858503021299839,\n      -0.0998515784740448,\n      -0.1542772501707077,\n      -0.0470099039375782,\n      -0.12248252332210541,\n      -0.15803958475589752,\n      -0.09450267255306244,\n      0.09489660710096359,\n      0.18596765398979187,\n      0.10402277112007141,\n      -0.28702250123023987,\n      -0.33180755376815796,\n      -0.20281799137592316,\n      -0.06972038745880127,\n      -0.03837069496512413,\n      -0.02239038050174713,\n      -0.07939201593399048,\n      0.040755368769168854,\n      -0.04678730294108391,\n      -0.01703268103301525,\n      0.20631742477416992,\n      0.2041403204202652,\n      -0.11710700392723083,\n      0.5981282591819763,\n      -0.09409008175134659,\n      0.1276732236146927,\n      -0.05574595183134079,\n      -0.08588626980781555,\n      0.07564660161733627,\n      0.09508813172578812,\n      -0.07919439673423767,\n      0.07502157241106033,\n      -0.13839483261108398,\n      -0.08790223300457001,\n      0.18147775530815125,\n      0.014002826064825058,\n      0.33050379157066345,\n      0.15746422111988068,\n      -0.0012312111211940646,\n      -0.01717819832265377\n    ],\n    [\n      -0.049337729811668396,\n      -0.25081348419189453,\n      -0.09281174093484879,\n      0.007727284450083971,\n      0.19251953065395355,\n      0.013646545819938183,\n      -0.09981174021959305,\n      -0.45988261699676514,\n      -0.09433955699205399,\n      -0.02321835234761238,\n      0.00871738325804472,\n      -0.2401668131351471,\n      -0.29249924421310425,\n      0.033145297318696976,\n      0.08899916708469391,\n      0.06436201184988022,\n      0.22819466888904572,\n      0.8446778059005737,\n      -0.0030743919778615236,\n      -0.06640619784593582,\n      0.004932293202728033,\n      -0.47813647985458374,\n      -0.01945245824754238,\n      -0.6641075611114502,\n      -0.10394883900880814,\n      -0.23606987297534943,\n      -0.04755371809005737,\n      -0.3163021504878998,\n      0.0927969366312027,\n      -0.1444014608860016,\n      -0.15632379055023193,\n      0.1263875514268875,\n      -0.02288808859884739,\n      0.23840370774269104,\n      -0.4262140989303589,\n      0.03808695450425148,\n      -0.2587064206600189,\n      -0.10681041330099106,\n      -0.27944910526275635,\n      -0.012667844071984291,\n      -0.04578428342938423,\n      0.07030569761991501,\n      0.0029238443821668625,\n      -0.2608352601528168,\n      -0.1478152722120285,\n      0.193078875541687,\n      -0.046339958906173706,\n      0.35504114627838135,\n      -0.16476699709892273,\n      0.10348972678184509\n    ],\n    [\n      -1.026081919670105,\n      0.02744203247129917,\n      0.09528150409460068,\n      0.3015398681163788,\n      0.19705185294151306,\n      0.11942197382450104,\n      0.06681142002344131,\n      0.0752897709608078,\n      0.21040531992912292,\n      0.10917877405881882,\n      0.17000837624073029,\n      -0.4611847996711731,\n      0.08706481754779816,\n      -0.34832972288131714,\n      0.04933488368988037,\n      -0.03971042484045029,\n      -0.13147708773612976,\n      -0.04960452765226364,\n      0.14727677404880524,\n      0.4749727249145508,\n      0.20106777548789978,\n      0.049444835633039474,\n      -0.04854841157793999,\n      0.207233726978302,\n      -0.049859073013067245,\n      0.19830238819122314,\n      -0.13150238990783691,\n      -0.1400122493505478,\n      -0.07341067492961884,\n      -0.14692482352256775,\n      -0.3291547894477844,\n      -0.18517127633094788,\n      0.26848304271698,\n      -0.28378555178642273,\n      0.013217049650847912,\n      0.2950667142868042,\n      0.026300951838493347,\n      -0.06482067704200745,\n      -0.15643450617790222,\n      0.08075056970119476,\n      -0.15444286167621613,\n      0.09519573301076889,\n      0.26808351278305054,\n      -0.19517827033996582,\n      0.12917156517505646,\n      -0.1733582466840744,\n      -0.15164172649383545,\n      0.24183572828769684,\n      -0.054356083273887634,\n      -0.08053381741046906\n    ],\n    [\n      0.2168618142604828,\n      -0.20145684480667114,\n      0.19428397715091705,\n      -0.7246829867362976,\n      -0.2696896195411682,\n      0.13537590205669403,\n      0.16436220705509186,\n      0.05112558975815773,\n      -0.45426303148269653,\n      -0.019151555374264717,\n      -0.20496296882629395,\n      -0.27425745129585266,\n      -0.1241728886961937,\n      -0.04212280362844467,\n      -0.20894964039325714,\n      -0.2541345953941345,\n      -0.11864986270666122,\n      -0.22503788769245148,\n      0.41987642645835876,\n      -0.025292381644248962,\n      -0.19198943674564362,\n      0.36040639877319336,\n      -0.06132618337869644,\n      0.049134623259305954,\n      -0.27169284224510193,\n      -0.13051527738571167,\n      -0.1766655296087265,\n      -0.25108131766319275,\n      -0.11183443665504456,\n      -0.2144920825958252,\n      0.4685809314250946,\n      -0.21427913010120392,\n      -0.035761263221502304,\n      0.2656055688858032,\n      -0.16916987299919128,\n      0.24911807477474213,\n      -0.01632564887404442,\n      0.014032711274921894,\n      -0.20321834087371826,\n      -0.026093054562807083,\n      -0.20576433837413788,\n      0.1716836392879486,\n      -0.38304486870765686,\n      0.174935981631279,\n      -0.03438805788755417,\n      0.14550459384918213,\n      -0.057037413120269775,\n      -0.08195150643587112,\n      -0.0025741150602698326,\n      -0.16669988632202148\n    ],\n    [\n      0.23142458498477936,\n      -0.07066720724105835,\n      0.17238746583461761,\n      0.29519540071487427,\n      -0.19976314902305603,\n      0.04449260234832764,\n      0.27449196577072144,\n      0.02509736828505993,\n      -0.2874855101108551,\n      -0.048564452677965164,\n      -0.0011683930642902851,\n      -0.36487382650375366,\n      0.1721363067626953,\n      -0.33491894602775574,\n      0.25553280115127563,\n      -0.20792138576507568,\n      0.0665774717926979,\n      -0.41052478551864624,\n      -0.4131990969181061,\n      -0.018855014815926552,\n      0.3250945508480072,\n      -0.0759248211979866,\n      0.026377229019999504,\n      -0.6710410714149475,\n      -0.013546440750360489,\n      0.06632640957832336,\n      -0.3221264183521271,\n      0.2857840955257416,\n      -0.14754991233348846,\n      -0.03256852552294731,\n      -0.14526675641536713,\n      0.09109043329954147,\n      0.15627263486385345,\n      -0.21006415784358978,\n      -0.18224649131298065,\n      -0.06908787786960602,\n      0.2840081751346588,\n      -0.20591585338115692,\n      -0.2587043344974518,\n      -0.15888836979866028,\n      0.21626555919647217,\n      0.22565510869026184,\n      -0.6376015543937683,\n      0.25216415524482727,\n      0.4058956801891327,\n      -0.07704833894968033,\n      -0.19777317345142365,\n      0.10293310135602951,\n      0.054722681641578674,\n      0.006313395220786333\n    ],\n    [\n      0.10213359445333481,\n      0.09516975283622742,\n      -0.05086249113082886,\n      -0.6481035947799683,\n      -0.01004539243876934,\n      0.20505502820014954,\n      0.4609759449958801,\n      -0.5429484844207764,\n      -0.4750175476074219,\n      -0.03546210750937462,\n      -0.034196190536022186,\n      -0.3148801326751709,\n      0.1869203746318817,\n      -0.31235817074775696,\n      -0.1983237862586975,\n      0.22295261919498444,\n      -0.14915519952774048,\n      -0.1021757423877716,\n      -0.011998548172414303,\n      0.11987872421741486,\n      -0.03372648358345032,\n      -0.10986010730266571,\n      0.21892975270748138,\n      0.20584993064403534,\n      0.023383690044283867,\n      -0.024683650583028793,\n      -0.2989889085292816,\n      0.098348468542099,\n      -0.17126165330410004,\n      0.03279683366417885,\n      -0.2868504524230957,\n      -0.038162991404533386,\n      -0.2008214294910431,\n      -0.7928566932678223,\n      -0.3392408490180969,\n      0.16560392081737518,\n      0.4668349623680115,\n      -0.136211559176445,\n      -0.044954363256692886,\n      0.3486177921295166,\n      -0.04184315726161003,\n      0.13062907755374908,\n      -0.019696103408932686,\n      0.1324857473373413,\n      0.34107741713523865,\n      0.09835230559110641,\n      -0.01589975319802761,\n      -0.389157235622406,\n      -0.10209089517593384,\n      0.14743943512439728\n    ],\n    [\n      -0.45345762372016907,\n      0.29428210854530334,\n      -0.2090853899717331,\n      0.18112409114837646,\n      -0.285805344581604,\n      0.251838743686676,\n      0.28611043095588684,\n      0.43320441246032715,\n      0.3690531849861145,\n      -0.09351328015327454,\n      0.0426195003092289,\n      0.34546083211898804,\n      0.1765691488981247,\n      -0.0577813945710659,\n      -0.02116667665541172,\n      -0.09047780930995941,\n      -0.06086895242333412,\n      0.08966435492038727,\n      0.10113221406936646,\n      0.05189082771539688,\n      0.5182246565818787,\n      -0.22100670635700226,\n      0.14233289659023285,\n      -0.16081075370311737,\n      0.22065068781375885,\n      -0.04606064409017563,\n      -0.41266435384750366,\n      0.22386234998703003,\n      -0.17480097711086273,\n      0.04691757261753082,\n      -0.2166384607553482,\n      0.03212357684969902,\n      0.18761563301086426,\n      0.8363586664199829,\n      -0.1254202127456665,\n      0.4455370008945465,\n      0.042582012712955475,\n      0.16312097012996674,\n      0.3006892800331116,\n      -0.2718108594417572,\n      0.3281973600387573,\n      0.19491545855998993,\n      -0.18445897102355957,\n      -0.20376981794834137,\n      -0.2981553077697754,\n      0.1237783282995224,\n      -0.016201067715883255,\n      0.1662474125623703,\n      -0.10616820305585861,\n      -0.06032990664243698\n    ],\n    [\n      0.232689768075943,\n      -0.44447171688079834,\n      0.2105448693037033,\n      0.2751316428184509,\n      0.37711894512176514,\n      0.0022084673400968313,\n      0.36341285705566406,\n      -0.12637849152088165,\n      0.12386828660964966,\n      -0.33580708503723145,\n      -0.27682510018348694,\n      -0.19002771377563477,\n      -0.3964892327785492,\n      -0.3058200776576996,\n      -0.3354426920413971,\n      -0.26642197370529175,\n      -0.35868096351623535,\n      -0.21975313127040863,\n      0.18821504712104797,\n      -0.0631081610918045,\n      0.3153245747089386,\n      0.228838250041008,\n      -0.42645883560180664,\n      -0.3963789641857147,\n      -0.22596678137779236,\n      -0.39264315366744995,\n      -0.3945613205432892,\n      -0.18992874026298523,\n      -0.23827767372131348,\n      -0.4351137578487396,\n      0.14640508592128754,\n      0.2626323699951172,\n      0.05601993203163147,\n      0.17204931378364563,\n      -0.4240603744983673,\n      0.24811267852783203,\n      -0.3620324432849884,\n      -0.2059416025876999,\n      -0.38578566908836365,\n      -0.3957683742046356,\n      -0.37723562121391296,\n      0.43035218119621277,\n      0.36406394839286804,\n      0.19895757734775543,\n      0.36992719769477844,\n      -0.06417641043663025,\n      -0.24774883687496185,\n      0.3493569493293762,\n      -0.308956503868103,\n      -0.29132598638534546\n    ],\n    [\n      0.023923726752400398,\n      -0.2246474176645279,\n      0.05102603882551193,\n      -0.05412611365318298,\n      0.6693800091743469,\n      0.15467675030231476,\n      0.2673344016075134,\n      0.4409731328487396,\n      0.218567356467247,\n      -0.027865683659911156,\n      0.14328183233737946,\n      0.28424784541130066,\n      0.29170235991477966,\n      -0.21605022251605988,\n      -0.21891926229000092,\n      -0.20754027366638184,\n      -0.06515808403491974,\n      -0.45035234093666077,\n      0.34284478425979614,\n      -0.011246146634221077,\n      -0.565503716468811,\n      0.5686787366867065,\n      0.1872691512107849,\n      0.22016026079654694,\n      -0.054653093218803406,\n      0.14171342551708221,\n      0.25563979148864746,\n      0.33700719475746155,\n      -0.11873351037502289,\n      -0.41830453276634216,\n      -0.006842146161943674,\n      -0.05337260663509369,\n      0.0794617310166359,\n      0.4183383882045746,\n      -0.1572146862745285,\n      0.11208879202604294,\n      -0.1934702843427658,\n      0.06295482814311981,\n      -0.2973204255104065,\n      -0.24784885346889496,\n      0.21634313464164734,\n      -0.06762319058179855,\n      -0.06652755290269852,\n      0.34584861993789673,\n      0.15847043693065643,\n      -0.4551164507865906,\n      0.2029569298028946,\n      -0.18294475972652435,\n      0.16255755722522736,\n      0.11765027046203613\n    ],\n    [\n      0.14456596970558167,\n      0.23278626799583435,\n      -0.1887637823820114,\n      0.3462575674057007,\n      0.04277247190475464,\n      -0.3237273097038269,\n      -0.001447975286282599,\n      -0.11434292048215866,\n      0.1814809888601303,\n      -0.0034667898435145617,\n      0.20990003645420074,\n      0.13823778927326202,\n      0.07381826639175415,\n      -0.07411574572324753,\n      -0.004729114938527346,\n      0.05571029707789421,\n      0.09177210181951523,\n      0.18506179749965668,\n      -0.27223801612854004,\n      -0.08937279134988785,\n      -0.1868707537651062,\n      -0.5123598575592041,\n      0.06821846216917038,\n      -0.061055004596710205,\n      0.08442848920822144,\n      0.04776329547166824,\n      0.190947026014328,\n      0.022976884618401527,\n      -0.08878858387470245,\n      0.061760641634464264,\n      0.27804508805274963,\n      0.1212826669216156,\n      -0.0813269391655922,\n      -0.787715494632721,\n      0.032655902206897736,\n      -0.14507944881916046,\n      -0.0301596038043499,\n      -0.01838349923491478,\n      -0.004320202860981226,\n      0.1694989651441574,\n      0.0779040977358818,\n      -0.038277775049209595,\n      0.9786800742149353,\n      0.21260057389736176,\n      0.26448294520378113,\n      0.364663690328598,\n      0.09841179847717285,\n      0.6356847882270813,\n      -0.08115611970424652,\n      -0.01330023817718029\n    ],\n    [\n      -0.07180869579315186,\n      0.10873803496360779,\n      -0.32323765754699707,\n      -0.09696871787309647,\n      0.11804728955030441,\n      0.11073006689548492,\n      -0.22454044222831726,\n      -0.6546792387962341,\n      0.3506810963153839,\n      0.11831827461719513,\n      0.05820620432496071,\n      -0.09232763200998306,\n      0.08361367136240005,\n      -0.15400183200836182,\n      0.06318897008895874,\n      -0.12331978976726532,\n      -0.21589307487010956,\n      0.1472158432006836,\n      0.03838353604078293,\n      -0.05095091834664345,\n      0.31625866889953613,\n      -0.460136741399765,\n      0.09728768467903137,\n      -0.5579326152801514,\n      -0.09198839217424393,\n      0.019890688359737396,\n      0.10566385835409164,\n      -0.21438159048557281,\n      -0.16338294744491577,\n      -0.1149531677365303,\n      0.14449386298656464,\n      0.3294515013694763,\n      0.1777021735906601,\n      -0.0473497100174427,\n      -0.05907529592514038,\n      -0.33387672901153564,\n      0.07428231835365295,\n      -0.06336923688650131,\n      -0.06709934771060944,\n      0.231694757938385,\n      0.15215325355529785,\n      -0.1876620352268219,\n      -0.03445301577448845,\n      -0.28921061754226685,\n      -0.11027219146490097,\n      -0.27413409948349,\n      -0.14854860305786133,\n      0.13366009294986725,\n      -0.10490302741527557,\n      0.03806979954242706\n    ],\n    [\n      -0.542121410369873,\n      -0.09144406765699387,\n      0.06516508758068085,\n      0.1444881111383438,\n      -0.06035938858985901,\n      -0.18947480618953705,\n      0.18588626384735107,\n      0.20526662468910217,\n      0.20760725438594818,\n      0.09160985052585602,\n      0.07479687035083771,\n      -0.0952373743057251,\n      0.014119399711489677,\n      0.12985841929912567,\n      -0.025724386796355247,\n      -0.08784901350736618,\n      0.001289311796426773,\n      0.055360712110996246,\n      -0.08706086874008179,\n      0.08450601994991302,\n      0.14436344802379608,\n      -0.014574314467608929,\n      0.13801687955856323,\n      -0.5366510152816772,\n      0.1834602653980255,\n      0.06558286398649216,\n      -0.15803036093711853,\n      0.12210821360349655,\n      -0.05941721424460411,\n      -0.0416509173810482,\n      -0.5011538863182068,\n      0.18266136944293976,\n      0.013269172050058842,\n      0.09535439312458038,\n      -0.11875949800014496,\n      0.044719498604536057,\n      -0.06809011846780777,\n      0.17947758734226227,\n      -0.026595115661621094,\n      0.0039175753481686115,\n      0.03470650687813759,\n      0.11196630448102951,\n      0.04789390787482262,\n      -0.02402217872440815,\n      -0.031177038326859474,\n      0.181244894862175,\n      -0.20908616483211517,\n      -0.04465540125966072,\n      0.01718798093497753,\n      0.008290152065455914\n    ],\n    [\n      -0.7346735000610352,\n      -0.658176600933075,\n      0.20461003482341766,\n      0.07439437508583069,\n      -0.13856419920921326,\n      0.07014510780572891,\n      -0.24387702345848083,\n      0.11632256209850311,\n      0.8128634691238403,\n      -0.04128701612353325,\n      -0.21861222386360168,\n      -0.09075810015201569,\n      0.023668358102440834,\n      0.18390357494354248,\n      -0.04334521293640137,\n      -0.13654561340808868,\n      0.07880350947380066,\n      -0.5477664470672607,\n      0.3222714066505432,\n      0.20942415297031403,\n      0.24103082716464996,\n      0.0010369870578870177,\n      0.07903929054737091,\n      -0.7326748967170715,\n      -0.018193775787949562,\n      0.14304082095623016,\n      -0.5774063467979431,\n      0.04663686826825142,\n      -0.38176771998405457,\n      -0.09514370560646057,\n      -0.2693597674369812,\n      -0.1760627031326294,\n      0.10530556738376617,\n      -0.0075754462741315365,\n      -0.2862960696220398,\n      -0.12947608530521393,\n      -0.340509295463562,\n      0.31726163625717163,\n      0.16130729019641876,\n      0.4510969817638397,\n      -0.1292114555835724,\n      0.5841554403305054,\n      -0.1322399079799652,\n      0.135287806391716,\n      -0.2676231265068054,\n      0.038900963962078094,\n      0.28827574849128723,\n      0.1366054266691208,\n      0.17256863415241241,\n      -0.053452786058187485\n    ],\n    [\n      -0.3961889147758484,\n      -0.46040841937065125,\n      0.2196914255619049,\n      0.05046434327960014,\n      -0.33147358894348145,\n      0.2684059143066406,\n      0.0474977120757103,\n      -0.017679188400506973,\n      0.26660850644111633,\n      -0.09793423116207123,\n      0.1641201674938202,\n      0.10482963919639587,\n      0.22193235158920288,\n      0.10818701237440109,\n      -0.10824785381555557,\n      -0.019321439787745476,\n      -0.014227773062884808,\n      -0.020141875371336937,\n      -0.13751612603664398,\n      0.05705909803509712,\n      -0.22966720163822174,\n      -0.18884046375751495,\n      0.20037731528282166,\n      -0.29322007298469543,\n      0.0446658581495285,\n      0.07818875461816788,\n      -0.19028626382350922,\n      0.13077665865421295,\n      0.08423267304897308,\n      0.008018983528017998,\n      -0.519278883934021,\n      0.21920281648635864,\n      0.05410536378622055,\n      0.2532445192337036,\n      -0.20290900766849518,\n      0.22744540870189667,\n      -0.1427524983882904,\n      0.13327495753765106,\n      -0.05179793760180473,\n      0.006791737861931324,\n      0.09608040004968643,\n      0.20773813128471375,\n      0.24394957721233368,\n      0.1281476765871048,\n      -0.026160025969147682,\n      0.03990846127271652,\n      -0.24915944039821625,\n      0.0083148293197155,\n      0.2173263132572174,\n      -0.05482521653175354\n    ],\n    [\n      -0.46897947788238525,\n      0.1939813494682312,\n      -0.032292917370796204,\n      0.31936466693878174,\n      -0.13258635997772217,\n      0.08417251706123352,\n      0.06938806176185608,\n      -0.5968083143234253,\n      0.15931163728237152,\n      0.025160523131489754,\n      0.10951870679855347,\n      -0.06244724616408348,\n      -0.028845395892858505,\n      0.0747387632727623,\n      0.10975819081068039,\n      0.024257972836494446,\n      0.14591236412525177,\n      0.18429380655288696,\n      -0.33624523878097534,\n      0.33360517024993896,\n      -0.24695807695388794,\n      -0.4387824237346649,\n      0.0842091292142868,\n      -0.03296291083097458,\n      0.14091207087039948,\n      -0.004438221920281649,\n      0.06744446605443954,\n      -0.06671259552240372,\n      -0.0782000795006752,\n      0.03303195908665657,\n      -0.25879067182540894,\n      0.09065626561641693,\n      0.12398450076580048,\n      0.9408542513847351,\n      0.03293769806623459,\n      0.09764484316110611,\n      -0.0036100498400628567,\n      -0.09164755046367645,\n      0.10436680167913437,\n      0.0855497345328331,\n      0.10866373777389526,\n      0.0606394000351429,\n      -0.15805990993976593,\n      0.04706179350614548,\n      -0.18564823269844055,\n      0.32456475496292114,\n      0.013767565600574017,\n      0.2760608196258545,\n      -0.08011768013238907,\n      0.15603870153427124\n    ],\n    [\n      0.06530220061540604,\n      -0.1938244104385376,\n      -0.06791539490222931,\n      0.1684546023607254,\n      0.588080108165741,\n      -0.05347514525055885,\n      -0.027186203747987747,\n      -0.6285409927368164,\n      -0.03389909118413925,\n      -0.07645681500434875,\n      -0.1571633219718933,\n      0.3467240035533905,\n      -0.08898109942674637,\n      -0.1689881533384323,\n      0.024456139653921127,\n      -0.25249725580215454,\n      -0.31211522221565247,\n      0.08157378435134888,\n      -0.0979478731751442,\n      -0.12501373887062073,\n      -0.25109243392944336,\n      -0.15055960416793823,\n      -0.13126854598522186,\n      0.07118936628103256,\n      0.01431370247155428,\n      0.10147762298583984,\n      0.08022072166204453,\n      -0.18964862823486328,\n      -0.10489154607057571,\n      -0.31562647223472595,\n      -0.20392489433288574,\n      0.09224116057157516,\n      -0.06302554905414581,\n      0.40176886320114136,\n      -0.17208777368068695,\n      0.05694785341620445,\n      -0.13730579614639282,\n      -0.10423211753368378,\n      -0.15937544405460358,\n      -0.046164944767951965,\n      0.04765899106860161,\n      0.0626683160662651,\n      -0.42633363604545593,\n      -0.019328445196151733,\n      0.23325718939304352,\n      0.0571424625813961,\n      -0.1589375138282776,\n      0.5786423087120056,\n      0.02490766905248165,\n      -0.13667245209217072\n    ],\n    [\n      -0.28771018981933594,\n      0.11562525480985641,\n      0.13038471341133118,\n      0.05946330353617668,\n      0.23543591797351837,\n      0.24393689632415771,\n      -0.227728009223938,\n      -0.2421497106552124,\n      0.25773248076438904,\n      0.009055891074240208,\n      0.042949821799993515,\n      -0.13164646923542023,\n      0.01785208471119404,\n      0.032690197229385376,\n      0.08964651823043823,\n      -0.2653658986091614,\n      -0.2937764823436737,\n      -0.04746067896485329,\n      -0.0561523400247097,\n      -0.6565371155738831,\n      0.12242861092090607,\n      0.2353164255619049,\n      -0.11213594675064087,\n      0.10395673662424088,\n      -0.02287227101624012,\n      -0.11630058288574219,\n      0.10504776239395142,\n      -0.0014501643599942327,\n      -0.007164937909692526,\n      -0.022275321185588837,\n      -0.1653028130531311,\n      0.022580260410904884,\n      0.2609707713127136,\n      -0.7206534147262573,\n      -0.2215133160352707,\n      -0.5918362140655518,\n      -0.0035360658075660467,\n      -0.1185218021273613,\n      0.07292405515909195,\n      -0.2265658974647522,\n      -0.05880394205451012,\n      0.17773087322711945,\n      0.5721856355667114,\n      0.14834612607955933,\n      -0.011880542151629925,\n      0.16113513708114624,\n      -0.2819710969924927,\n      0.14223423600196838,\n      -0.11073827743530273,\n      -0.14627468585968018\n    ],\n    [\n      0.040821321308612823,\n      -0.20160777866840363,\n      -0.05570583418011665,\n      0.004754761699587107,\n      -0.5616325736045837,\n      -0.05090862140059471,\n      0.1670302450656891,\n      -0.19239507615566254,\n      -0.48343104124069214,\n      -0.17791041731834412,\n      -0.22411267459392548,\n      -0.047452736645936966,\n      -0.1549903005361557,\n      -0.11515803635120392,\n      -0.08002974838018417,\n      -0.11467361450195312,\n      -0.1606924831867218,\n      -0.21562559902668,\n      0.23508183658123016,\n      0.15265709161758423,\n      -0.2348959594964981,\n      0.008862434886395931,\n      -0.04552668705582619,\n      -0.2892385721206665,\n      -0.0220014750957489,\n      -0.07863172143697739,\n      -0.0968034639954567,\n      -0.028949622064828873,\n      -0.12328556925058365,\n      -0.17934443056583405,\n      -0.11296600848436356,\n      -0.11576605588197708,\n      0.022461242973804474,\n      0.3400169312953949,\n      -0.2392335832118988,\n      -0.17141087353229523,\n      -0.13378208875656128,\n      -0.21247243881225586,\n      -0.1951640099287033,\n      -0.15454521775245667,\n      0.04504670202732086,\n      0.09864577651023865,\n      0.13875499367713928,\n      0.3617118299007416,\n      -0.05337328463792801,\n      0.32986173033714294,\n      0.32204610109329224,\n      0.26012060046195984,\n      -0.01748303696513176,\n      -0.1285756379365921\n    ],\n    [\n      -0.36107540130615234,\n      -0.23316925764083862,\n      -0.0005261289770714939,\n      0.23076489567756653,\n      0.43435776233673096,\n      -0.07070977240800858,\n      -0.3080746531486511,\n      0.44530999660491943,\n      -0.21137793362140656,\n      -0.003932056948542595,\n      -0.01840481534600258,\n      0.14063875377178192,\n      -0.08314060419797897,\n      -0.16481882333755493,\n      0.004022188019007444,\n      -0.07609113305807114,\n      -0.03963321074843407,\n      0.6156153082847595,\n      0.11877110600471497,\n      -0.5687031745910645,\n      -0.35158517956733704,\n      -0.36379000544548035,\n      0.23329587280750275,\n      0.23173342645168304,\n      0.300642192363739,\n      0.08068472146987915,\n      -0.08311239629983902,\n      -0.2587467133998871,\n      0.12286464869976044,\n      -0.05061759427189827,\n      0.015050970017910004,\n      0.31135866045951843,\n      -0.14364568889141083,\n      0.2513780891895294,\n      -0.0556335411965847,\n      -0.27911168336868286,\n      -0.12411349266767502,\n      -0.02942589484155178,\n      -0.05207731947302818,\n      0.2911379337310791,\n      0.0683738961815834,\n      0.11167559027671814,\n      -0.09840988367795944,\n      0.12265720218420029,\n      -0.08435030281543732,\n      -0.16036835312843323,\n      -0.11171884834766388,\n      0.33031022548675537,\n      -0.09275522083044052,\n      -0.035640649497509\n    ],\n    [\n      0.3510441184043884,\n      -0.035518430173397064,\n      0.1440327763557434,\n      -0.006552477832883596,\n      -0.19958235323429108,\n      -0.08374685049057007,\n      0.2199011594057083,\n      -0.811407208442688,\n      -0.04289533570408821,\n      0.13828562200069427,\n      0.07134454697370529,\n      -0.5139862298965454,\n      0.020516257733106613,\n      -0.03293876349925995,\n      0.17269454896450043,\n      0.21152111887931824,\n      0.16204194724559784,\n      0.23592056334018707,\n      -0.16266214847564697,\n      0.0677967369556427,\n      0.31394630670547485,\n      0.14992383122444153,\n      0.0345202200114727,\n      -0.4013928771018982,\n      0.03696933388710022,\n      -0.1786099076271057,\n      -0.07471805810928345,\n      0.2389640063047409,\n      0.05816379189491272,\n      0.05141192674636841,\n      0.09221195429563522,\n      0.07856535166501999,\n      0.0026714038103818893,\n      -0.36936143040657043,\n      -0.18525612354278564,\n      -0.5446624159812927,\n      0.1133439689874649,\n      -0.09709855169057846,\n      -0.10296037048101425,\n      0.26518115401268005,\n      0.16124682128429413,\n      -0.005081284791231155,\n      -0.6138672828674316,\n      0.16156505048274994,\n      0.3464154005050659,\n      0.1769765019416809,\n      -0.38684895634651184,\n      -0.5983367562294006,\n      -0.026411525905132294,\n      0.07318852841854095\n    ],\n    [\n      -0.7498357892036438,\n      0.21063445508480072,\n      -0.20563901960849762,\n      0.12044817209243774,\n      -0.04553386941552162,\n      0.2572336792945862,\n      0.20674782991409302,\n      -0.06195385009050369,\n      0.18859727680683136,\n      0.007550694048404694,\n      -0.08987760543823242,\n      -0.027995649725198746,\n      0.05885208025574684,\n      -0.07259200513362885,\n      -0.06576356291770935,\n      -0.07104557752609253,\n      -0.0628618597984314,\n      0.4630751609802246,\n      0.12426687031984329,\n      -0.11534039676189423,\n      0.09168915450572968,\n      -0.2538411319255829,\n      -0.030915604904294014,\n      0.08649176359176636,\n      -0.14069928228855133,\n      0.0226445235311985,\n      0.20883843302726746,\n      0.042885906994342804,\n      0.0958562046289444,\n      -0.0032906290143728256,\n      -0.2657560110092163,\n      0.041179344058036804,\n      0.3829485774040222,\n      0.07227688282728195,\n      0.09059078246355057,\n      0.13119883835315704,\n      -0.09330447763204575,\n      -0.33125561475753784,\n      -0.2283739298582077,\n      0.09303531050682068,\n      -0.18821220099925995,\n      0.06556814163923264,\n      0.3424741327762604,\n      0.1392085701227188,\n      -0.03409351781010628,\n      0.2728283703327179,\n      0.14966948330402374,\n      0.3203960657119751,\n      -0.17096266150474548,\n      -0.010720270685851574\n    ],\n    [\n      0.274734228849411,\n      0.07605774700641632,\n      0.05234622582793236,\n      0.1289052963256836,\n      0.27108338475227356,\n      -0.13771401345729828,\n      0.19547876715660095,\n      -0.1534997969865799,\n      0.4250090718269348,\n      0.015761446207761765,\n      0.1377006471157074,\n      -0.05734637379646301,\n      -0.014811448752880096,\n      -0.21452313661575317,\n      -0.08140499144792557,\n      0.08328989148139954,\n      0.09046290814876556,\n      -0.19140201807022095,\n      0.07533324509859085,\n      0.14998066425323486,\n      0.24585434794425964,\n      0.15308083593845367,\n      -0.1229090616106987,\n      0.18071803450584412,\n      0.25310906767845154,\n      -0.24070844054222107,\n      -0.010410037823021412,\n      0.06309834122657776,\n      -0.1297897845506668,\n      -0.16657744348049164,\n      0.37421631813049316,\n      0.2173442393541336,\n      -0.2538306415081024,\n      -0.8775382041931152,\n      -0.07950269430875778,\n      -0.15970011055469513,\n      0.10895555466413498,\n      0.008314157836139202,\n      0.14638255536556244,\n      0.072003573179245,\n      0.6190196871757507,\n      0.02677847445011139,\n      0.2699536085128784,\n      -0.19142070412635803,\n      -0.2592925727367401,\n      -0.2565824091434479,\n      -0.262987345457077,\n      -0.1489066481590271,\n      -0.0432773195207119,\n      -0.08126294612884521\n    ],\n    [\n      -0.6639822125434875,\n      0.0880877748131752,\n      -0.005751533899456263,\n      -0.19255366921424866,\n      0.3057144582271576,\n      -0.08999431878328323,\n      -0.019433338195085526,\n      -0.1559668332338333,\n      -0.15749216079711914,\n      -0.011301230639219284,\n      -0.11821027845144272,\n      -0.30145376920700073,\n      0.006112752482295036,\n      -0.1560450941324234,\n      -0.5127982497215271,\n      0.40875670313835144,\n      0.13342377543449402,\n      -0.04711974039673805,\n      0.27859559655189514,\n      -0.25916099548339844,\n      -0.10510764271020889,\n      -0.041911106556653976,\n      0.037033870816230774,\n      -0.20566396415233612,\n      0.05327753722667694,\n      -0.1296263337135315,\n      -0.32234781980514526,\n      0.0389278419315815,\n      -0.40813302993774414,\n      -0.02087150141596794,\n      -0.19934184849262238,\n      -0.07899387180805206,\n      -0.25175032019615173,\n      -0.2088051736354828,\n      0.13817232847213745,\n      -0.04268897697329521,\n      -0.42443978786468506,\n      -0.4097912311553955,\n      -0.04090137407183647,\n      -0.15951122343540192,\n      0.04449190944433212,\n      0.21488967537879944,\n      0.12114228308200836,\n      0.09004763513803482,\n      -0.006397236604243517,\n      -0.07948800921440125,\n      0.19370833039283752,\n      0.683956503868103,\n      -0.11428768187761307,\n      -0.10317038744688034\n    ],\n    [\n      -0.5914733409881592,\n      -0.4672893285751343,\n      0.12114158272743225,\n      0.00020536499505396932,\n      -0.2679958939552307,\n      0.26662468910217285,\n      0.1092640832066536,\n      -0.21656334400177002,\n      0.17207543551921844,\n      0.08223824948072433,\n      0.001051013940013945,\n      0.0034190842416137457,\n      0.13344737887382507,\n      -0.5096179842948914,\n      -0.006541278213262558,\n      0.08032097667455673,\n      -0.005550340749323368,\n      -0.04771978780627251,\n      -0.21063093841075897,\n      0.052590880542993546,\n      0.2812643349170685,\n      0.15709182620048523,\n      -0.13493511080741882,\n      -0.48999109864234924,\n      0.15144944190979004,\n      -0.10234226286411285,\n      0.2640151381492615,\n      0.18310357630252838,\n      0.17227308452129364,\n      0.111046701669693,\n      -0.5058703422546387,\n      0.07171519845724106,\n      0.02478756196796894,\n      -0.045091744512319565,\n      -0.2756809890270233,\n      -0.28741583228111267,\n      -0.22533218562602997,\n      -0.33518844842910767,\n      -0.46407124400138855,\n      0.41094958782196045,\n      0.031681012362241745,\n      0.19160377979278564,\n      0.22435687482357025,\n      0.21299022436141968,\n      0.07534463703632355,\n      0.3006431758403778,\n      0.4345487356185913,\n      0.170183464884758,\n      0.19991670548915863,\n      0.0319303423166275\n    ],\n    [\n      -0.21427439153194427,\n      -0.20036062598228455,\n      0.07794656604528427,\n      -0.14284713566303253,\n      0.25140538811683655,\n      0.3854665160179138,\n      0.46422120928764343,\n      0.36879846453666687,\n      -0.362587034702301,\n      -0.17855657637119293,\n      -0.19077561795711517,\n      -0.2539205253124237,\n      -0.1934506893157959,\n      -0.2961663603782654,\n      -0.08020146191120148,\n      -0.1223592534661293,\n      -0.09612061083316803,\n      -0.08487864583730698,\n      0.030782759189605713,\n      0.13439634442329407,\n      -0.016591235995292664,\n      -0.23079997301101685,\n      -0.3025470972061157,\n      0.2626320719718933,\n      -0.09485653042793274,\n      -0.09040284156799316,\n      -0.31883135437965393,\n      -0.09886917471885681,\n      0.06538306176662445,\n      -0.15228867530822754,\n      -0.1643630415201187,\n      -0.31876298785209656,\n      0.13331247866153717,\n      0.09882406890392303,\n      -0.07007796317338943,\n      0.28392279148101807,\n      -0.2196052223443985,\n      -0.3272660970687866,\n      -0.08426976948976517,\n      -0.20013859868049622,\n      -0.3208335340023041,\n      0.014314942993223667,\n      -0.2237250953912735,\n      0.12013121694326401,\n      -0.0037638030480593443,\n      0.3212618827819824,\n      -0.2214321345090866,\n      0.1737927347421646,\n      -0.24851001799106598,\n      -0.2322719395160675\n    ],\n    [\n      -0.11294197291135788,\n      0.22274744510650635,\n      -0.2330183982849121,\n      -0.7324397563934326,\n      0.4676364064216614,\n      -0.07979191094636917,\n      0.7319750785827637,\n      0.12972930073738098,\n      0.021024459972977638,\n      0.36666375398635864,\n      0.23946106433868408,\n      0.02072736620903015,\n      0.2554115653038025,\n      -0.04969428852200508,\n      -0.3307410478591919,\n      0.18114912509918213,\n      0.028270814567804337,\n      0.015179718844592571,\n      -0.12330888956785202,\n      -0.06808768212795258,\n      0.11298701167106628,\n      0.3111850619316101,\n      0.15453025698661804,\n      -0.036064568907022476,\n      -0.15355663001537323,\n      -0.08854296803474426,\n      -0.1556462049484253,\n      0.13265253603458405,\n      -0.1301099956035614,\n      0.1300511360168457,\n      0.05666785687208176,\n      0.05629177764058113,\n      0.02590491622686386,\n      0.4094301164150238,\n      -0.044398609548807144,\n      0.452364057302475,\n      0.3546171188354492,\n      -0.13299521803855896,\n      -0.1892523318529129,\n      0.25428497791290283,\n      0.04711531475186348,\n      0.021106703206896782,\n      0.6154316067695618,\n      -0.22931945323944092,\n      -0.16165129840373993,\n      -0.11139032244682312,\n      0.09658266603946686,\n      -0.1307058334350586,\n      -0.049249663949012756,\n      0.32638129591941833\n    ],\n    [\n      -0.46581366658210754,\n      0.03137717396020889,\n      -0.1218806728720665,\n      0.09832312911748886,\n      -0.009367070160806179,\n      -0.11927109211683273,\n      0.7112864851951599,\n      -0.734951376914978,\n      -0.07221429795026779,\n      -0.011268037371337414,\n      0.04569047689437866,\n      0.04369886964559555,\n      -0.032484911382198334,\n      0.09794636815786362,\n      -0.32328906655311584,\n      0.1170458123087883,\n      0.12185964733362198,\n      0.19422496855258942,\n      0.17055435478687286,\n      -0.003093091771006584,\n      -0.5446919202804565,\n      0.12102081626653671,\n      0.027012798935174942,\n      0.17547154426574707,\n      0.055674947798252106,\n      -0.13497957587242126,\n      0.17319534718990326,\n      -0.07802064716815948,\n      -0.19643627107143402,\n      0.04205043241381645,\n      0.213498055934906,\n      0.07426552474498749,\n      0.02582940086722374,\n      0.451744019985199,\n      -0.022335831075906754,\n      0.3766533136367798,\n      -0.4091492295265198,\n      -0.015331956557929516,\n      -0.05918412283062935,\n      -0.06600836664438248,\n      0.14513570070266724,\n      -0.035239510238170624,\n      0.40115639567375183,\n      -0.20591595768928528,\n      -0.23105399310588837,\n      -0.07920513302087784,\n      -0.16204708814620972,\n      0.0754992663860321,\n      0.07525797933340073,\n      0.09106867760419846\n    ],\n    [\n      0.33886078000068665,\n      -0.15836355090141296,\n      0.11545507609844208,\n      -0.307790070772171,\n      -0.3583011329174042,\n      -0.4114040434360504,\n      -0.17187818884849548,\n      -0.5699397325515747,\n      0.1770302802324295,\n      0.05155625194311142,\n      0.25689783692359924,\n      -0.07145075500011444,\n      -0.39431020617485046,\n      0.13408873975276947,\n      -0.047830283641815186,\n      -0.3556029498577118,\n      -0.1249995082616806,\n      0.33150044083595276,\n      0.05355924740433693,\n      0.0580989271402359,\n      -0.15124715864658356,\n      0.1805446445941925,\n      -0.27744537591934204,\n      -0.05419614911079407,\n      -0.14440882205963135,\n      0.1770683228969574,\n      0.24545757472515106,\n      -0.026452191174030304,\n      -0.008723985403776169,\n      0.038196153938770294,\n      0.03161676228046417,\n      0.1373031884431839,\n      -0.4143252372741699,\n      0.3969142436981201,\n      -0.06792732328176498,\n      -0.1232592761516571,\n      -0.16646811366081238,\n      0.06960832327604294,\n      0.03154929354786873,\n      -0.123887799680233,\n      -0.46353864669799805,\n      -0.3144764304161072,\n      -0.42628389596939087,\n      0.5393332242965698,\n      0.2493637055158615,\n      -0.12768633663654327,\n      -0.0760553702712059,\n      -0.5405345559120178,\n      -0.08851554989814758,\n      0.03100821003317833\n    ],\n    [\n      -0.9735671877861023,\n      -0.04137355089187622,\n      0.35126975178718567,\n      -0.10205425322055817,\n      -0.025996064767241478,\n      -0.0580744668841362,\n      -0.1881583034992218,\n      0.16081131994724274,\n      0.13075855374336243,\n      0.07091133296489716,\n      -0.006204509641975164,\n      0.11568878591060638,\n      -0.08243140578269958,\n      0.07040645182132721,\n      0.09325511008501053,\n      0.028905700892210007,\n      0.15887966752052307,\n      0.10187280178070068,\n      0.15174755454063416,\n      0.35554617643356323,\n      -0.3071753978729248,\n      -0.1548262983560562,\n      0.1684771478176117,\n      0.27995574474334717,\n      0.003711925121024251,\n      -0.06057242304086685,\n      0.02376418001949787,\n      -0.05478191003203392,\n      -0.1576126664876938,\n      -0.2858216464519501,\n      -0.2798856496810913,\n      0.2505143880844116,\n      0.23728787899017334,\n      -0.2534143328666687,\n      -0.2322133481502533,\n      -0.5443998575210571,\n      -0.04497148469090462,\n      0.07133816182613373,\n      0.0013389517553150654,\n      0.08474922180175781,\n      0.08366569876670837,\n      -0.0008644083864055574,\n      -0.08997055888175964,\n      0.12731218338012695,\n      0.24673467874526978,\n      -0.10185850411653519,\n      0.3059315085411072,\n      -0.06534984707832336,\n      -0.04846072942018509,\n      -0.01469405833631754\n    ],\n    [\n      0.4039386510848999,\n      -0.13471591472625732,\n      0.027628807350993156,\n      -0.034723374992609024,\n      0.17345017194747925,\n      0.08883119374513626,\n      -0.609667956829071,\n      -0.6436949968338013,\n      0.182982936501503,\n      0.042495593428611755,\n      -0.024463780224323273,\n      0.05978408828377724,\n      0.03751043230295181,\n      -0.029150333255529404,\n      0.009013879112899303,\n      -0.08349311351776123,\n      -0.22251936793327332,\n      -0.14870254695415497,\n      0.14848963916301727,\n      -0.20075109601020813,\n      0.16211113333702087,\n      -0.7619535326957703,\n      0.18761400878429413,\n      0.2754133343696594,\n      -0.023325514048337936,\n      -0.2032531052827835,\n      0.08467560261487961,\n      -0.2319343388080597,\n      -0.05421333387494087,\n      -0.14198003709316254,\n      -0.12662072479724884,\n      0.1968347579240799,\n      -0.03838423267006874,\n      0.3028755784034729,\n      0.20501698553562164,\n      0.24259786307811737,\n      0.010480734519660473,\n      -0.13123713433742523,\n      0.09066292643547058,\n      0.013406556099653244,\n      -0.14184661209583282,\n      -0.008003261871635914,\n      -0.49010083079338074,\n      -0.42149728536605835,\n      0.05045846849679947,\n      0.16863785684108734,\n      -0.34566766023635864,\n      0.38461145758628845,\n      0.04986937716603279,\n      0.01650077849626541\n    ],\n    [\n      -0.12281874567270279,\n      -0.06224335357546806,\n      0.01965164951980114,\n      0.036817919462919235,\n      0.01696583442389965,\n      0.08764713257551193,\n      0.025815490633249283,\n      -0.17511172592639923,\n      -0.013873397372663021,\n      -0.0496794767677784,\n      -0.01390866655856371,\n      0.2167406976222992,\n      -0.016514839604496956,\n      -0.13962915539741516,\n      -0.12483185529708862,\n      -0.055758003145456314,\n      -0.11738940328359604,\n      0.02215925231575966,\n      0.11241894215345383,\n      0.1515226513147354,\n      -0.053121067583560944,\n      -0.027850231155753136,\n      -0.030644873157143593,\n      -0.014299597591161728,\n      0.09084217250347137,\n      -0.006216452457010746,\n      -0.2358647584915161,\n      -0.15631726384162903,\n      0.015916109085083008,\n      -0.16677819192409515,\n      -0.01726796291768551,\n      -0.10054439306259155,\n      0.026055416092276573,\n      0.3712286651134491,\n      -0.3131727874279022,\n      0.18314386904239655,\n      0.05813062936067581,\n      -0.014103861525654793,\n      -0.04529847577214241,\n      -0.09722699970006943,\n      0.11625651270151138,\n      0.18894419074058533,\n      -0.26961812376976013,\n      -0.05369789898395538,\n      -0.11172289401292801,\n      0.0071302978321909904,\n      -0.09918048977851868,\n      0.24546149373054504,\n      -0.06613007187843323,\n      -0.05349305644631386\n    ],\n    [\n      -0.10706257820129395,\n      -0.014513627626001835,\n      -0.01650029793381691,\n      -0.022345444187521935,\n      -0.2375548928976059,\n      0.14926046133041382,\n      0.05824356526136398,\n      -0.13587728142738342,\n      -0.5190252065658569,\n      -0.23528948426246643,\n      -0.19684772193431854,\n      0.08985518664121628,\n      -0.09019802510738373,\n      -0.16498494148254395,\n      -0.16400064527988434,\n      -0.17665381729602814,\n      -0.1495702862739563,\n      0.04063289985060692,\n      0.018178608268499374,\n      0.0679873526096344,\n      -0.16714423894882202,\n      -0.036891527473926544,\n      -0.10445253551006317,\n      -0.3927890360355377,\n      -0.17704632878303528,\n      -0.1166594922542572,\n      0.012740793637931347,\n      -0.14439982175827026,\n      -0.1996820718050003,\n      0.10451915115118027,\n      -0.03469640761613846,\n      -0.0004551775346044451,\n      -0.07949463278055191,\n      -0.039038173854351044,\n      -0.15445175766944885,\n      -0.1380963772535324,\n      -0.1047859638929367,\n      0.033395104110240936,\n      -0.1391402930021286,\n      -0.11430581659078598,\n      -0.16138480603694916,\n      -0.013521285727620125,\n      0.16289760172367096,\n      0.06026143580675125,\n      -0.08880285918712616,\n      0.27378812432289124,\n      -0.7299013137817383,\n      0.12981179356575012,\n      -0.21602566540241241,\n      -0.03437846899032593\n    ],\n    [\n      -0.07186418026685715,\n      -0.055654894560575485,\n      0.023081880062818527,\n      -0.014171147719025612,\n      0.3718494772911072,\n      0.15714910626411438,\n      0.5212790966033936,\n      -0.057541623711586,\n      -0.1241057738661766,\n      0.03125312924385071,\n      -0.07598546147346497,\n      -0.046080268919467926,\n      0.2779327929019928,\n      -0.012316126376390457,\n      -0.2595909833908081,\n      -0.1340722143650055,\n      0.05720413103699684,\n      -0.5329489707946777,\n      0.11871278285980225,\n      0.2923130691051483,\n      -0.11652947962284088,\n      0.443386971950531,\n      0.11399097740650177,\n      -0.39724382758140564,\n      0.0910022184252739,\n      0.07382900267839432,\n      -0.02331758663058281,\n      0.2986155152320862,\n      -0.17429904639720917,\n      -0.02758517861366272,\n      -0.005745948292315006,\n      -0.06663986295461655,\n      -0.03176482766866684,\n      0.5119363069534302,\n      0.04511744901537895,\n      0.4174656867980957,\n      -0.2300683706998825,\n      -0.01011301577091217,\n      0.21833020448684692,\n      -0.052029334008693695,\n      0.1804760843515396,\n      0.4034663438796997,\n      0.2590266168117523,\n      -0.2084108591079712,\n      0.06783833354711533,\n      -0.13150788843631744,\n      0.14485538005828857,\n      0.017271699383854866,\n      -0.05304720252752304,\n      -0.057500772178173065\n    ],\n    [\n      -0.6604385375976562,\n      -0.06986501812934875,\n      0.15083815157413483,\n      0.2219560742378235,\n      0.3681388199329376,\n      -0.24392996728420258,\n      -0.2625875174999237,\n      0.9658703804016113,\n      -0.06555946171283722,\n      -0.04877181351184845,\n      -0.09979092329740524,\n      0.19463306665420532,\n      -0.27455735206604004,\n      0.13402371108531952,\n      -0.37881940603256226,\n      0.10094164311885834,\n      0.13876503705978394,\n      -0.447234183549881,\n      -0.1345466673374176,\n      0.12663979828357697,\n      -0.24984702467918396,\n      -0.035014744848012924,\n      -0.039633940905332565,\n      -0.210527703166008,\n      0.20277711749076843,\n      0.2858479917049408,\n      0.08572372794151306,\n      0.036232221871614456,\n      -0.36387720704078674,\n      0.06315044313669205,\n      -0.30623769760131836,\n      -0.06741265207529068,\n      -0.30438026785850525,\n      0.09858912974596024,\n      0.21076439321041107,\n      0.40742477774620056,\n      -0.43515026569366455,\n      -0.157973974943161,\n      0.3636869788169861,\n      -0.24309775233268738,\n      -0.09181814640760422,\n      0.2706977427005768,\n      0.36064156889915466,\n      -0.036217812448740005,\n      -0.07993815094232559,\n      0.07931309938430786,\n      0.45539867877960205,\n      0.07486847043037415,\n      -0.024540740996599197,\n      -0.03841240331530571\n    ],\n    [\n      -0.307781845331192,\n      -0.034045323729515076,\n      -0.0005269395187497139,\n      -0.19310402870178223,\n      -0.026454007253050804,\n      0.18268531560897827,\n      -0.1673271805047989,\n      -0.09658391773700714,\n      -0.11760496348142624,\n      0.10150157660245895,\n      0.11351918429136276,\n      -0.13518212735652924,\n      -0.09026109427213669,\n      -0.018844613805413246,\n      -0.14594362676143646,\n      -0.09376624971628189,\n      0.06628540903329849,\n      0.16020053625106812,\n      -0.08841736614704132,\n      0.3251096308231354,\n      -0.06996630132198334,\n      -0.07898686081171036,\n      -0.010989238508045673,\n      0.1558353155851364,\n      -0.056911613792181015,\n      -0.23423999547958374,\n      0.18207210302352905,\n      -0.22104330360889435,\n      -0.017040472477674484,\n      -0.3210408091545105,\n      -0.15926975011825562,\n      -0.19055554270744324,\n      0.19050413370132446,\n      -0.10608169436454773,\n      -0.02338770218193531,\n      -0.12002378702163696,\n      -0.17946694791316986,\n      -0.14818662405014038,\n      -0.16118058562278748,\n      -0.08301952481269836,\n      0.09531703591346741,\n      0.13841426372528076,\n      -0.060536179691553116,\n      -0.032108038663864136,\n      0.12043174356222153,\n      0.047239068895578384,\n      -0.01471246499568224,\n      -0.08050220459699631,\n      0.07315975427627563,\n      -0.046004388481378555\n    ],\n    [\n      -0.28982484340667725,\n      -0.08965249359607697,\n      0.10088396072387695,\n      0.044090624898672104,\n      0.376592218875885,\n      -0.08558747172355652,\n      0.3574191927909851,\n      -0.7516403198242188,\n      0.22832489013671875,\n      0.05686384066939354,\n      -0.07454803586006165,\n      -0.24541698396205902,\n      -0.14552736282348633,\n      0.14130418002605438,\n      0.048876725137233734,\n      -0.12436597049236298,\n      0.15423622727394104,\n      0.20786859095096588,\n      0.2391236424446106,\n      0.2524341642856598,\n      -0.6358815431594849,\n      0.03661400079727173,\n      -0.08577727526426315,\n      -0.4446631073951721,\n      0.15542654693126678,\n      -0.15350627899169922,\n      -0.0007989830919541419,\n      0.07778619229793549,\n      0.030498310923576355,\n      -0.1517181098461151,\n      -0.061767566949129105,\n      0.07415859401226044,\n      0.10067164897918701,\n      0.6278400421142578,\n      -0.2585182189941406,\n      0.05506706237792969,\n      0.08913308382034302,\n      -0.08907200396060944,\n      0.11066343635320663,\n      -0.4256574809551239,\n      -0.09891491383314133,\n      0.10912012308835983,\n      -0.26256752014160156,\n      -0.2248126119375229,\n      0.03860471025109291,\n      0.09886275976896286,\n      0.17226631939411163,\n      -0.025798451155424118,\n      -0.09317366033792496,\n      -0.05684979632496834\n    ],\n    [\n      -0.37888529896736145,\n      -0.16662314534187317,\n      0.07877650856971741,\n      0.33243048191070557,\n      0.33421242237091064,\n      0.04229206219315529,\n      0.06698255240917206,\n      0.5107118487358093,\n      -0.19182775914669037,\n      -0.17274397611618042,\n      0.11606128513813019,\n      -0.03831028565764427,\n      0.06799456477165222,\n      -0.23126263916492462,\n      0.17431028187274933,\n      0.1489490568637848,\n      0.162456214427948,\n      -0.2882015109062195,\n      -0.2679458260536194,\n      0.0428243950009346,\n      -0.39375025033950806,\n      -0.2358098179101944,\n      0.29235535860061646,\n      0.15713715553283691,\n      0.1423434019088745,\n      0.25796201825141907,\n      0.2916123569011688,\n      0.13723468780517578,\n      0.15418045222759247,\n      -0.2943059206008911,\n      -0.4850558638572693,\n      0.3461304306983948,\n      0.1502530425786972,\n      -0.43227624893188477,\n      0.12890367209911346,\n      0.26796016097068787,\n      -0.11738845705986023,\n      0.1978311687707901,\n      -0.19749420881271362,\n      0.06924550980329514,\n      0.07652833312749863,\n      0.4045135974884033,\n      0.571014404296875,\n      0.07547017186880112,\n      -0.2980058193206787,\n      0.02181682363152504,\n      -0.3753744959831238,\n      0.25161999464035034,\n      -0.08938631415367126,\n      -0.03211525082588196\n    ],\n    [\n      -0.49257224798202515,\n      -0.09388156235218048,\n      0.0012350549222901464,\n      0.18059146404266357,\n      0.0011243124026805162,\n      -0.20821326971054077,\n      0.20082233846187592,\n      0.4148622155189514,\n      -0.06775477528572083,\n      -0.058037225157022476,\n      -0.10307089239358902,\n      -0.10048248618841171,\n      -0.05047061666846275,\n      0.12638187408447266,\n      -0.4556204080581665,\n      -0.46558642387390137,\n      -0.0772886872291565,\n      -0.6759362816810608,\n      -0.18581464886665344,\n      0.11267050355672836,\n      -0.15583279728889465,\n      -0.16388441622257233,\n      0.23114919662475586,\n      -0.3857767581939697,\n      -0.06667602807283401,\n      0.01988230086863041,\n      0.21008752286434174,\n      0.27807095646858215,\n      -0.41078582406044006,\n      -0.022135812789201736,\n      -0.3471928834915161,\n      0.20695865154266357,\n      -0.08093701303005219,\n      -0.22231993079185486,\n      -0.08899680525064468,\n      0.06807203590869904,\n      -0.25230348110198975,\n      -0.0014619325520470738,\n      0.08896080404520035,\n      -0.3568987548351288,\n      0.11678487062454224,\n      1.2827746868133545,\n      -0.014346606098115444,\n      0.08429359644651413,\n      -0.0919218435883522,\n      0.07523822784423828,\n      -0.04129438474774361,\n      0.11294715851545334,\n      0.24544107913970947,\n      -0.042737267911434174\n    ],\n    [\n      -0.48072704672813416,\n      -0.18810005486011505,\n      0.0374588705599308,\n      -0.05727287009358406,\n      -0.012722119688987732,\n      0.14065587520599365,\n      -0.003408705350011587,\n      0.1122223362326622,\n      0.340884804725647,\n      -0.15227937698364258,\n      -0.005189506337046623,\n      -0.04472450166940689,\n      -0.09916462749242783,\n      -0.040019359439611435,\n      0.16836167871952057,\n      0.1395205408334732,\n      -0.1028418242931366,\n      -0.30095595121383667,\n      0.48140761256217957,\n      0.21626786887645721,\n      0.09874799847602844,\n      0.05301765352487564,\n      -0.005831585265696049,\n      -0.436678022146225,\n      0.0865410789847374,\n      0.0398264117538929,\n      -0.16981326043605804,\n      0.038346730172634125,\n      -0.23007746040821075,\n      0.2817489802837372,\n      -0.09297100454568863,\n      0.11058783531188965,\n      -0.00528382183983922,\n      0.4117512106895447,\n      -0.06859137117862701,\n      0.03556973114609718,\n      -0.05617225542664528,\n      -0.08823215961456299,\n      0.14528052508831024,\n      0.020949963480234146,\n      0.12183349579572678,\n      -0.1382322609424591,\n      -0.13707837462425232,\n      0.0687415823340416,\n      -0.14030469954013824,\n      0.15818864107131958,\n      0.19207841157913208,\n      -0.033355679363012314,\n      0.000742139876820147,\n      0.12967880070209503\n    ],\n    [\n      0.3177640438079834,\n      0.03588578850030899,\n      -0.0939568430185318,\n      0.12949220836162567,\n      0.33398666977882385,\n      0.08892231434583664,\n      -0.31378835439682007,\n      -0.21445046365261078,\n      0.10949963331222534,\n      -0.20236003398895264,\n      -0.15028421580791473,\n      0.10194536298513412,\n      0.0360504575073719,\n      0.13285522162914276,\n      -0.0156784076243639,\n      -0.11452782899141312,\n      -0.10350515693426132,\n      -0.6414148807525635,\n      -0.047703564167022705,\n      0.1961110234260559,\n      -0.44564664363861084,\n      0.03627097234129906,\n      -0.18321439623832703,\n      -0.17838998138904572,\n      -0.13331592082977295,\n      0.1563485711812973,\n      -0.35932809114456177,\n      0.2402045577764511,\n      0.11359019577503204,\n      -0.3508605360984802,\n      0.16375738382339478,\n      0.08338475227355957,\n      -0.006938163191080093,\n      0.1444127857685089,\n      -0.05632803216576576,\n      0.03932121396064758,\n      0.10900809615850449,\n      0.07841342687606812,\n      0.005295031704008579,\n      -0.15297219157218933,\n      0.081606924533844,\n      0.8683913946151733,\n      -0.3636198043823242,\n      0.00883118249475956,\n      0.2409229576587677,\n      -0.35125502943992615,\n      -0.005632631480693817,\n      0.14590124785900116,\n      0.13412189483642578,\n      -0.20834799110889435\n    ],\n    [\n      0.0017151302890852094,\n      -0.305098295211792,\n      0.12967051565647125,\n      0.022289346903562546,\n      -0.8472644090652466,\n      0.03573717549443245,\n      0.3002815842628479,\n      -0.27122384309768677,\n      -0.4793834686279297,\n      -0.1459033042192459,\n      -0.20546512305736542,\n      -0.06842319667339325,\n      -0.08808693289756775,\n      0.07788810133934021,\n      -0.0466180220246315,\n      -0.10641411691904068,\n      0.10383602976799011,\n      -0.07716105133295059,\n      -0.06759563088417053,\n      0.3445044755935669,\n      0.3319224417209625,\n      0.1663070172071457,\n      0.17388753592967987,\n      -0.10807190835475922,\n      -0.15468557178974152,\n      -0.029798904433846474,\n      0.16725780069828033,\n      -0.048754703253507614,\n      -0.1269257366657257,\n      0.2151537537574768,\n      0.035467348992824554,\n      0.03292301297187805,\n      0.052482858300209045,\n      0.11522477865219116,\n      -0.4186718165874481,\n      0.07275981456041336,\n      0.07813485711812973,\n      -0.15219397842884064,\n      -0.1822328418493271,\n      -0.040962252765893936,\n      0.016940148547291756,\n      -0.07688132673501968,\n      0.023374469950795174,\n      0.32189542055130005,\n      0.1331385225057602,\n      -0.15390019118785858,\n      0.4464007318019867,\n      -0.3328644931316376,\n      -0.19362740218639374,\n      -0.08363735675811768\n    ]\n  ],\n  "hidden_bias": [\n    -0.0910271480679512,\n    0.07835296541452408,\n    -0.2877812087535858,\n    0.536487877368927,\n    -0.17930996417999268,\n    -0.4639956057071686,\n    -0.3537112772464752,\n    -0.20710493624210358,\n    -0.012712390162050724,\n    -0.05994794890284538,\n    -0.35677650570869446,\n    0.2527293562889099,\n    -0.01686500385403633,\n    0.23655222356319427,\n    -0.47310569882392883,\n    0.045338720083236694,\n    -0.6184362173080444,\n    -0.04151638597249985,\n    -0.4233519434928894,\n    -0.0897008553147316,\n    -0.6524513363838196,\n    0.09297261387109756,\n    -0.29338133335113525,\n    0.05763170123100281,\n    -0.07583652436733246,\n    -0.04625171050429344,\n    0.09338902682065964,\n    -0.1357330083847046,\n    -0.6487828493118286,\n    -0.41045859456062317,\n    0.29976391792297363,\n    0.009535040706396103,\n    -0.41884297132492065,\n    -0.04188937693834305,\n    -0.1835736483335495,\n    -0.5297504663467407,\n    -0.12637421488761902,\n    0.023102406412363052,\n    0.19732075929641724,\n    0.12233787775039673,\n    -0.039070453494787216,\n    -0.22084969282150269,\n    -0.23959358036518097,\n    -0.033917754888534546,\n    -0.23289041221141815,\n    0.26288798451423645,\n    -0.11982260644435883,\n    0.30594393610954285,\n    -0.6035798788070679,\n    0.015034908428788185,\n    0.07078143209218979,\n    0.08575760573148727,\n    0.04642843082547188,\n    0.11488718539476395,\n    0.13566158711910248,\n    -0.16577203571796417,\n    0.02040993794798851,\n    -0.0511690191924572,\n    -0.17587392032146454,\n    -0.5083130598068237,\n    -0.38909322023391724,\n    -0.11265306919813156,\n    -0.039799291640520096,\n    -0.036090556532144547\n  ],\n  "output_weight": [\n    0.2131989598274231,\n    0.1977451592683792,\n    0.10760799050331116,\n    -0.1919630616903305,\n    0.1623319536447525,\n    -0.15136662125587463,\n    0.18045802414417267,\n    -0.30183181166648865,\n    -0.008809132501482964,\n    0.15819451212882996,\n    0.2064265012741089,\n    0.14527276158332825,\n    -0.12433601915836334,\n    0.08363532274961472,\n    -0.25438863039016724,\n    0.22195687890052795,\n    0.26992323994636536,\n    0.1499648541212082,\n    0.1286066770553589,\n    0.3266310393810272,\n    0.347166508436203,\n    -0.17328840494155884,\n    0.13646559417247772,\n    0.17151620984077454,\n    0.12186865508556366,\n    -0.2395879328250885,\n    0.1403948813676834,\n    -0.20059140026569366,\n    -0.2968544065952301,\n    -0.4102231562137604,\n    -0.12591375410556793,\n    0.2557772397994995,\n    -0.3231998383998871,\n    -0.1460215002298355,\n    0.0783180370926857,\n    0.4334082007408142,\n    0.09213857352733612,\n    -0.22668281197547913,\n    0.1472545862197876,\n    -0.2056395411491394,\n    0.10779472440481186,\n    0.17486685514450073,\n    -0.23726321756839752,\n    -0.13073530793190002,\n    -0.18027904629707336,\n    -0.12480378895998001,\n    0.22221282124519348,\n    0.11064167320728302,\n    -0.4147959351539612,\n    -0.24766920506954193,\n    -0.2846771478652954,\n    -0.1911633461713791,\n    0.22691060602664948,\n    0.0041246572509408,\n    0.0713290274143219,\n    0.11663182824850082,\n    -0.24798740446567535,\n    0.024134783074259758,\n    -0.22529403865337372,\n    0.3082430362701416,\n    0.11048208177089691,\n    0.07865811884403229,\n    0.06063178554177284,\n    0.1315339207649231\n  ],\n  "output_bias": 0.1393638551235199,\n  "training": {\n    "method": "paired counterfactual residual policy distillation",\n    "seed": 20261020,\n    "loss_type": "ranking",\n    "regularization": 0.01,\n    "best_epoch": 97,\n    "validation_preference_loss": 0.46406126022338867,\n    "baseline_preference_loss": 0.7314271330833435,\n    "training_games": 2400,\n    "validation_games": 600,\n    "source_dataset": {\n      "method": "paired one-step policy improvement, phase-stratified whole-game reservoirs",\n      "input_features": [\n        "batch_size",\n        "truthful",\n        "bluff",\n        "bluff_pile",\n        "bluff_call_rate",\n        "bluff_finish",\n        "remaining_groups",\n        "remaining_largest_group",\n        "claimed_rank_held",\n        "bias",\n        "revealed_bluff_rate",\n        "rank_held",\n        "claim_count",\n        "pile_size",\n        "opponent_finishing",\n        "own_small_hand",\n        "opponent_small_hand",\n        "samples",\n        "bluff_any_call_probability",\n        "bluff_impossible_for_caller",\n        "truthful_turns_removed",\n        "bluff_known_pile_fraction",\n        "truthful_finish",\n        "bluff_no_impossibility_call_possible",\n        "placer_global_revealed_bluff_rate",\n        "known_outside_rank_fraction",\n        "known_pile_fraction",\n        "claim_fraction_of_original_hand",\n        "finishing_large_batch",\n        "known_placer_rank_before_claim",\n        "bluff_call_probability_batch",\n        "bluff_call_probability_pile",\n        "bluff_call_probability_finish",\n        "bluff_nearest_opponent_finish",\n        "truthful_call_probability_batch",\n        "bluff_remaining_truthful_turns",\n        "global_revealed_bluff_finishing",\n        "count_revealed_bluff_pile",\n        "global_revealed_bluff_pile",\n        "count_revealed_bluff_small_own_hand",\n        "global_revealed_bluff_count",\n        "opponent_hand_pace_relative_to_own",\n        "bluff_pooled_caller_probability",\n        "bluff_pooled_caller_uncertainty",\n        "bluff_specific_caller_uncertainty",\n        "bluff_sparse_own_outcomes",\n        "truthful_pooled_caller_probability_batch",\n        "bluff_pooled_caller_probability_pile",\n        "global_revealed_bluff_uncertainty",\n        "count_revealed_bluff_uncertainty"\n      ],\n      "feature_policy": "v5",\n      "games": 3000,\n      "states": 12000,\n      "actions": 67757,\n      "trials": 4,\n      "samples_per_phase": 2,\n      "max_actions": 10,\n      "seed_start": 50000000,\n      "incumbent": {\n        "format": "triad-bluff-linear-v2",\n        "features": [\n          "batch_size",\n          "truthful",\n          "bluff",\n          "bluff_pile",\n          "bluff_call_rate",\n          "bluff_finish",\n          "remaining_groups",\n          "remaining_largest_group",\n          "claimed_rank_held",\n          "bias",\n          "revealed_bluff_rate",\n          "rank_held",\n          "claim_count",\n          "pile_size",\n          "opponent_finishing",\n          "own_small_hand",\n          "opponent_small_hand",\n          "samples",\n          "bluff_any_call_probability",\n          "bluff_impossible_for_caller",\n          "truthful_turns_removed",\n          "bluff_known_pile_fraction",\n          "truthful_finish",\n          "bluff_no_impossibility_call_possible",\n          "placer_global_revealed_bluff_rate",\n          "known_outside_rank_fraction",\n          "known_pile_fraction",\n          "claim_fraction_of_original_hand",\n          "finishing_large_batch",\n          "known_placer_rank_before_claim"\n        ],\n        "weights": [\n          0.4755931924211193,\n          5.820676470779726,\n          -3.445920468550993,\n          0.5025301512894197,\n          -10.204982725387449,\n          -10.63672367105182,\n          1.9122839363624424,\n          6.692718730179745,\n          3.3398384247884447,\n          -3.967913294968817,\n          7.18093879068249,\n          -8.248647532051631,\n          -4.252488827029067,\n          -5.717390427387336,\n          -10.297751467694116,\n          -4.172529040138095,\n          3.643993344291045,\n          4.785033532440487,\n          -8.90940656773177,\n          -9.18268199666696,\n          3.5590525036076692,\n          -1.533025684746697,\n          5.957038223067108,\n          9.714173152584005,\n          3.5582377937169647,\n          -0.47803421898358756,\n          -1.2639464162072949,\n          0.6168380283723138,\n          2.861924827690982,\n          -7.115720155350792\n        ],\n        "training": {\n          "method": "archive-league cross-entropy policy optimization",\n          "seed": 20261016,\n          "generations": 30,\n          "population": 32,\n          "train_blocks": 8,\n          "train_horizon": 200,\n          "evaluation_horizon": 1000,\n          "validation_blocks": 64,\n          "selected_validation_points": 4.0859375,\n          "incumbent_validation_points": 3.62109375,\n          "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n        }\n      },\n      "opponent_pool": [\n        {\n          "kind": "v2",\n          "weights": [\n            0.4755931924211193,\n            5.820676470779726,\n            -3.445920468550993,\n            0.5025301512894197,\n            -10.204982725387449,\n            -10.63672367105182,\n            1.9122839363624424,\n            6.692718730179745,\n            3.3398384247884447,\n            -3.967913294968817,\n            7.18093879068249,\n            -8.248647532051631,\n            -4.252488827029067,\n            -5.717390427387336,\n            -10.297751467694116,\n            -4.172529040138095,\n            3.643993344291045,\n            4.785033532440487,\n            -8.90940656773177,\n            -9.18268199666696,\n            3.5590525036076692,\n            -1.533025684746697,\n            5.957038223067108,\n            9.714173152584005,\n            3.5582377937169647,\n            -0.47803421898358756,\n            -1.2639464162072949,\n            0.6168380283723138,\n            2.861924827690982,\n            -7.115720155350792\n          ]\n        },\n        {\n          "kind": "v1",\n          "weights": [\n            4.351868833686492,\n            5.6322712663248415,\n            -2.6260873498626167,\n            0.7063932700439369,\n            -10.84185815552825,\n            -8.300750779890834,\n            -0.7486711333834275,\n            1.2260583122844886,\n            -1.5575700886723332,\n            -4.558178606791033,\n            8.25750430388175,\n            0.43873901259948533,\n            -1.3556124209301408,\n            -1.1333529834957075,\n            0.14766041336362645,\n            -1.5734330345566885,\n            -0.4730921181099686,\n            -0.5811911507935986\n          ]\n        },\n        {\n          "kind": "starter"\n        },\n        {\n          "kind": "competitive"\n        },\n        {\n          "kind": "random"\n        },\n        {\n          "kind": "aggressive"\n        },\n        {\n          "kind": "heuristic"\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.1,\n          "call_rate": 0.0\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.1,\n          "call_rate": 0.35\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.35,\n          "call_rate": 0.1\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.65,\n          "call_rate": 0.35\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.95,\n          "call_rate": 0.0\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.95,\n          "call_rate": 0.35\n        }\n      ],\n      "sdk_game_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6",\n      "hidden_state_use": "training returns only; dealt hands fixed within state, common future actor seeds across actions",\n      "future_draw_pile_shuffle": false,\n      "own_hand_size_histogram": {\n        "2": 640,\n        "3": 1392,\n        "4": 1021,\n        "5": 1355,\n        "6": 783,\n        "7": 1373,\n        "8": 606,\n        "9": 1081,\n        "10": 720,\n        "11": 1005,\n        "12": 98,\n        "13": 1443,\n        "14": 51,\n        "15": 65,\n        "16": 65,\n        "17": 41,\n        "18": 46,\n        "19": 31,\n        "20": 35,\n        "21": 27,\n        "22": 14,\n        "23": 21,\n        "24": 19,\n        "25": 14,\n        "26": 9,\n        "27": 8,\n        "28": 5,\n        "29": 4,\n        "30": 4,\n        "31": 5,\n        "32": 3,\n        "34": 3,\n        "35": 4,\n        "37": 1,\n        "38": 2,\n        "39": 1,\n        "40": 1,\n        "41": 1,\n        "43": 2,\n        "46": 1\n      },\n      "phase_histogram": {\n        "PLACE": 6000,\n        "CHALLENGE": 6000\n      },\n      "seconds": 2063.4\n    },\n    "base_score_factor": 0.25,\n    "hidden_state_use": "training labels only; policy input is own/public features",\n    "residual_scale": 0.5\n  }\n}\n')
(work/'previous.json').write_text('{\n  "format": "triad-bluff-linear-v2",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim"\n  ],\n  "weights": [\n    0.4755931924211193,\n    5.820676470779726,\n    -3.445920468550993,\n    0.5025301512894197,\n    -10.204982725387449,\n    -10.63672367105182,\n    1.9122839363624424,\n    6.692718730179745,\n    3.3398384247884447,\n    -3.967913294968817,\n    7.18093879068249,\n    -8.248647532051631,\n    -4.252488827029067,\n    -5.717390427387336,\n    -10.297751467694116,\n    -4.172529040138095,\n    3.643993344291045,\n    4.785033532440487,\n    -8.90940656773177,\n    -9.18268199666696,\n    3.5590525036076692,\n    -1.533025684746697,\n    5.957038223067108,\n    9.714173152584005,\n    3.5582377937169647,\n    -0.47803421898358756,\n    -1.2639464162072949,\n    0.6168380283723138,\n    2.861924827690982,\n    -7.115720155350792\n  ],\n  "training": {\n    "method": "archive-league cross-entropy policy optimization",\n    "seed": 20261016,\n    "generations": 30,\n    "population": 32,\n    "train_blocks": 8,\n    "train_horizon": 200,\n    "evaluation_horizon": 1000,\n    "validation_blocks": 64,\n    "selected_validation_points": 4.0859375,\n    "incumbent_validation_points": 3.62109375,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
sdk=Path("/kaggle/working/bluffSDK")
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/bluffSDK",str(sdk)],check=True)
subprocess.run(["git","-C",str(sdk),"checkout","--quiet","96da8042c285af4e4f43655f2ad99592bd49b8e6"],check=True)
os.environ["BLUFF_SDK"]=str(sdk)
subprocess.run(["python",str(work/"collect_bluff_counterfactual.py"),"--incumbent",str(work/"incumbent.json"),"--previous",str(work/"previous.json"),"--games","3500","--samples-per-phase","2","--trials","6","--max-actions","10","--workers","4","--seed-start","87000000","--source-compositions","balanced","--output","/kaggle/working/output"],check=True,timeout=21600)
